# 🪙 05 · Learning from data: likelihood, MLE and MAP
### Supervised vs unsupervised learning as probability, the likelihood, maximum-likelihood and maximum-a-posteriori estimation, priors as imaginary counts, and the two-step recipe of the whole course

**BiologyToML** · Level 1 · 02-620 core · Part A — Foundations: from biology to data · Module 05 of 44

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/05_likelihood_mle_map.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy (`scipy.stats`, `scipy.optimize`), pandas, Matplotlib |
| **Prerequisites** | Module 02 (random variables, Bernoulli/binomial/Gaussian, Bayes' rule) and Module 04 (derivatives, logs, setting the derivative to zero). Module 03 helps for Section 6. |
| **Estimated time** | 150 min (reading + running + the ✏️ exercises) |
| **02-620 topics covered** | Introduction to machine learning (the first lecture): supervised learning as learning $f: X \to Y$, regression vs classification and modelling uncertainty with $P(Y\mid X)$; unsupervised learning as density estimation $P(X)$; learning a joint distribution and conditioning it; MLE vs MAP and Bayes' rule in machine learning. From the naïve-Bayes lecture: maximum likelihood as counting (the coin-flip example), conditional vs joint likelihood, zero counts and MAP estimates with Beta/Dirichlet priors ("imaginary examples"). From the regression lecture: the two-step recipe (choose $P(Y\mid X;\theta)$, estimate θ by MLE or MAP) and the Gaussian prior that turns least squares into ridge regression |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> The leukaemia expression data (Sections 1, 3 and 9, with the original 38/34 training/test design) are
> **SIMULATED** with the toolkit's `simulate_leukemia_expression`, shaped like Golub *et al.* (1999) — the values are not
> the paper's. The drug-resistance isolates, the genotypes, the replicate data sets, the fold-change genes and the
> genotype–smoking–disease cohort are simulated as well and labelled as such. Two small worked examples of the 02-620
> lectures are typed in: the six-individual genotype table of the naïve-Bayes lecture and a classic eight-cell joint
> probability table of gender, working hours and wealth (from US census records).

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **State** the learning problem precisely — data $D$, unknown parameters $\theta$, a probability model — and **distinguish** supervised learning (model $P(Y\mid X)$) from unsupervised learning (model $P(X)$).
2. **Write** the likelihood and log-likelihood of independent observations and **compute** the maximum-likelihood estimate (MLE) by calculus and by grid search: the classic coin flips H,H,T,T,H,H,T ($\hat q = 4/7$) and an allele frequency from genotypes; **explain** why naïve-Bayes MLEs are count ratios.
3. **Derive** the Gaussian MLEs of a mean and a variance and **explain** why the MLE variance divides by $N$ (and is biased).
4. **Apply** Bayes' rule to parameters, **compute** a Beta posterior and the MAP estimate $(n_H + a - 1)/(N + a + b - 2)$, and **interpret** a prior as "imaginary examples" that cure zero counts.
5. **Compare** MLE and MAP by simulation as the sample size grows, using bias, variance and mean squared error.
6. **Show** that a Gaussian prior on a mean shrinks the MAP estimate toward the prior mean — the same formula as ridge regression.
7. **Learn** a joint distribution by counting and **condition** it to predict, and **quantify** why joint tables explode with the number of variables.
8. **Connect** everything to the two-step recipe (choose a model, estimate it by MLE or MAP) that structures every later module, and **evaluate** a fitted model on held-out test data.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit, simulate the leukaemia data | 5 min |
| 1 · The learning problem | supervised = $P(Y\mid X)$, unsupervised = $P(X)$, on leukaemia data | 15 min |
| 2 · Likelihood and the MLE | coin flips / resistant isolates, log-likelihood, allele frequencies, naïve-Bayes counts | 30 min |
| 3 · Gaussian MLE | mean and variance of a gene; the $N$ vs $N-1$ question | 15 min |
| 4 · Bayes' rule for parameters: MAP | Beta prior, posterior, imaginary counts, zero counts in the naïve-Bayes table | 25 min |
| 5 · MLE vs MAP as data accumulate | 20,000 replicate studies: bias, variance, MSE | 15 min |
| 6 · A Gaussian prior shrinks | fold changes from 3 replicates; the ridge connection | 10 min |
| 7 · Frequentist vs Bayesian | point estimates, credible intervals, prediction | 5 min |
| 8 · Joint distributions and conditioning | a census joint table; a genotype–smoking–disease cohort; the blow-up | 20 min |
| 9 · Generalisation and the two-step recipe | a one-gene generative classifier trained on 38, tested on 34 patients | 10 min |
| Wrap-up | takeaways, checkpoint, questions, lab note | 5 min |

## 🧫 The biological question

A hospital laboratory has just sequenced seven isolates of a bacterial pathogen and checked each one for a mutation
that confers resistance to a front-line antibiotic: **R, R, S, S, R, R, S** (R = resistant, S = susceptible). What is
the frequency of resistance in the population these isolates came from? "Four out of seven, 57 %" is the obvious
answer — and it is exactly the **maximum-likelihood estimate** of the classic coin-flip example of 02-620, where the
same sequence appears as coin flips H, H, T, T, H, H, T.

Now suppose only the first three isolates had been tested, and all three were susceptible. Is resistance absent?
The maximum-likelihood answer is "0 %", which no microbiologist would accept: surveillance from previous years says
resistance is rare but real. Combining such prior knowledge with new data is what **maximum a posteriori (MAP)**
estimation does, and it fixes the same zero-count problem that breaks naïve-Bayes classifiers (Module 08).

This notebook builds the two estimation principles that the first 02-620 lecture puts at the heart of the course —
MLE (frequentist, data only) and MAP (Bayesian, data plus prior belief) — and the probabilistic view of learning that
every later lecture uses: supervised learning as learning $P(Y\mid X)$, unsupervised learning as learning $P(X)$.

> 🧪 **Simulated data, with a known truth.** The course stores no data sets: every table in this notebook is generated
> by code. The leukaemia expression data come from the toolkit's `simulate_leukemia_expression`, which imitates the
> classic study of Golub *et al.* (1999) — 72 patients (47 ALL, 25 AML), 3,571 genes on a log10 scale, the original
> 38/34 training/test design and marker genes named after those that distinguished the two leukaemias in the paper.
> Simulated data have a second advantage: we know the truth (which genes really differ between the classes, by how
> much), so we can check whether an estimate found it. None of the numbers below are the paper's numbers.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
They generate the SIMULATED leukaemia expression data (shaped like Golub *et al.* 1999), simulate Hardy–Weinberg genotypes, provide the six-person genotype/cancer table of the naïve-Bayes lecture, and compute accuracy. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import numpy as np

NB_GENOTYPE_TABLE = np.array([
    [0, 0, 1, 1],
    [1, 0, 2, 1],
    [0, 2, 0, 1],
    [2, 0, 0, 0],
    [2, 1, 2, 0],
    [1, 2, 1, 0],
])


LEUKEMIA_MARKERS = {
    "CST3": ("AML", "cystatin C"), "ZYX": ("AML", "zyxin"), "CD33": ("AML", "CD33 antigen"),
    "MPO": ("AML", "myeloperoxidase"), "LYZ": ("AML", "lysozyme"), "CTSD": ("AML", "cathepsin D"),
    "FAH": ("AML", "fumarylacetoacetate hydrolase"), "LTC4S": ("AML", "leukotriene C4 synthase"),
    "LYN": ("AML", "LYN kinase"), "HOXA9": ("AML", "homeobox A9"), "CFD": ("AML", "adipsin"),
    "LEPR": ("AML", "leptin receptor"), "SRGN": ("AML", "proteoglycan 1 (serglycin)"),
    "CXCL8": ("AML", "interleukin-8"), "AZU1": ("AML", "azurocidin"), "MCL1": ("AML", "MCL1"),
    "ITGAX": ("AML", "CD11c"), "NFKBIA": ("AML", "MAD-3 (IκBα)"), "ELANE": ("AML", "neutrophil elastase"),
    "CEBPA": ("AML", "C/EBPα"), "CSF3R": ("AML", "G-CSF receptor"), "FCER1G": ("AML", "Fc receptor γ"),
    "TYROBP": ("AML", "DAP12"), "S100A9": ("AML", "calprotectin subunit"), "CFP": ("AML", "properdin"),
    "MYB": ("ALL", "c-Myb"), "CCND3": ("ALL", "cyclin D3"), "RBBP4": ("ALL", "RbAp48"),
    "SMARCA4": ("ALL", "SNF2 (BRG1)"), "TCF3": ("ALL", "E2A"), "DUT": ("ALL", "dUTPase"),
    "STMN1": ("ALL", "Op18 (stathmin)"), "IL7R": ("ALL", "IL-7 receptor"), "RABEP1": ("ALL", "rabaptin-5"),
    "CBX3": ("ALL", "heterochromatin protein p25"), "DNTT": ("ALL", "terminal deoxynucleotidyl transferase"),
    "CD24": ("ALL", "CD24"), "TOP2B": ("ALL", "topoisomerase II β"), "IRF2": ("ALL", "IRF-2"),
    "MYL6": ("ALL", "myosin light chain"), "PSMB8": ("ALL", "proteasome subunit"), "LMO2": ("ALL", "LMO2"),
    "CD79A": ("B-ALL", "MB-1 (Igα)"), "CD19": ("B-ALL", "CD19"), "CD79B": ("B-ALL", "Igβ"),
    "VPREB1": ("B-ALL", "pre-B receptor"), "PAX5": ("B-ALL", "PAX5"), "CD22": ("B-ALL", "CD22"),
    "CD3E": ("T-ALL", "CD3ε"), "CD3D": ("T-ALL", "CD3δ"), "CD2": ("T-ALL", "CD2"), "CD7": ("T-ALL", "CD7"),
    "LCK": ("T-ALL", "LCK kinase"), "CD1E": ("T-ALL", "CD1e"),
}


def simulate_leukemia_expression(seed: int = 1999) -> dict:
    """Synthetic leukaemia expression data shaped like the classic Golub et al. (1999) study.

    72 bone-marrow/blood samples — 47 acute lymphoblastic leukaemia (ALL: 38 B-lineage, 9 T-lineage)
    and 25 acute myeloid leukaemia (AML) — × 3,571 genes on a log10 microarray scale, with the
    original 38/34 training/test design (train: 19 B-ALL, 8 T-ALL, 11 AML).  Values below the
    detection limit are floored at 2.0 (≈ 23 % of entries) and saturate at 4.2.  The 54 genes of
    ``LEUKEMIA_MARKERS`` carry the strongest signals (42 differ between ALL and AML by 0.5–1.0
    log10 units; 12 mark B- or T-lineage ALL), 900 more genes differ modestly between ALL and AML,
    150 separate T-ALL from B-ALL, and 15 hidden co-expression programmes correlate genes.  Two
    AML samples are atypical (signature at 35 % strength).  The classes are cleaner than in the
    real study, so many classifiers make few or no errors.

    Returns a dict with ``X`` (72 × 3,571), ``y`` (0 = ALL, 1 = AML), ``labels``, ``subtype``,
    ``genes``, ``is_train``, ``sample_id`` (1–72), ``atypical`` (True for the two weak-signature AML
    samples), ``true_effect`` and ``source``.  ``true_effect`` is the planted AML − ALL difference
    per gene; it is 0 for the 12 lineage markers and the 150 T-vs-B genes, although the B-lineage
    markers still differ between ALL and AML through the subtype shifts (by about 0.8 log10 units).
    The default ``seed`` gives the course's standard data set.
    """
    rng = np.random.default_rng(seed)
    J = 3571
    subtype = np.array(["B-ALL"] * 19 + ["T-ALL"] * 8 + ["AML"] * 11 + ["B-ALL"] * 19 + ["T-ALL"] * 1 + ["AML"] * 14,
                       dtype=object)
    is_train = np.r_[np.ones(38, bool), np.zeros(34, bool)]
    order = np.r_[rng.permutation(38), 38 + rng.permutation(34)]          # mix classes within each set
    subtype, is_train = subtype[order], is_train[order]
    n = len(subtype)
    y = (subtype == "AML").astype(int)
    named = list(LEUKEMIA_MARKERS)
    genes = np.array(named + [f"G{j:04d}" for j in range(1, J - len(named) + 1)], dtype=object)
    low = rng.random(J) < 0.4                                             # genes mostly below detection
    mu = np.where(low, rng.uniform(1.5, 2.4, J), np.clip(rng.normal(2.9, 0.4, J), 2.2, 4.0))
    sd = rng.uniform(0.12, 0.35, J)
    X = mu + sd * rng.normal(0, 1, (n, J))
    factors = rng.normal(0, 1, (n, 15))                                  # co-expression programmes
    loadings = rng.normal(0, 0.15, (15, J)) * (rng.random((15, J)) < 0.08)
    X += factors @ loadings
    effect = np.zeros(J)
    strong = np.arange(len(named))
    for j, g in enumerate(named):
        cls = LEUKEMIA_MARKERS[g][0]
        size = rng.uniform(0.5, 1.0)
        effect[j] = size if cls == "AML" else -size if cls == "ALL" else 0.0
        mu[j] = max(mu[j], 2.5)
    moderate = rng.choice(np.arange(len(named), J), 900, replace=False)
    effect[moderate] = rng.choice([-1, 1], 900) * rng.uniform(0.1, 0.4, 900)
    strength = np.ones(n)
    strength[np.flatnonzero(y == 1)[rng.choice((y == 1).sum(), 2, replace=False)]] = 0.35   # atypical AML
    X[:, strong] = mu[strong] + sd[strong] * rng.normal(0, 1, (n, len(strong)))
    X += np.outer((y - y.mean()) * strength, effect)
    for g, (cls, _) in LEUKEMIA_MARKERS.items():
        if cls in ("B-ALL", "T-ALL"):
            j = named.index(g)
            X[:, j] += np.where(subtype == cls, 0.9, np.where(subtype == "AML", -0.2, -0.5))
    lineage = rng.choice(np.setdiff1d(np.arange(len(named), J), moderate), 150, replace=False)
    X[:, lineage] += np.outer(np.where(subtype == "T-ALL", 1.0, np.where(subtype == "B-ALL", -0.2, 0.0)),
                              rng.choice([-1, 1], 150) * rng.uniform(0.4, 0.8, 150))
    X = np.round(np.clip(X, 2.0, np.log10(16000)), 4)
    return dict(X=X, y=y, labels=np.where(y == 1, "AML", "ALL").astype(object), subtype=subtype, genes=genes,
                is_train=is_train, sample_id=np.arange(1, n + 1), atypical=strength < 1, true_effect=effect,
                source="SIMULATED leukaemia data (shaped like Golub et al. 1999)")


def simulate_genotypes(rng: np.random.Generator, n_samples: int, n_snps: int, n_pops: int = 1,
                       fst: float = 0.0, maf_range=(0.05, 0.5)):
    """Simulate 0/1/2 genotypes under Hardy–Weinberg equilibrium.

    Each SNP gets an ancestral allele frequency drawn uniformly from ``maf_range``.  With
    ``n_pops > 1`` and ``fst > 0`` every population's frequency is drawn from the
    Balding–Nichols model Beta(p(1−F)/F, (1−p)(1−F)/F).  Samples are split evenly across
    populations.  Returns ``(G, pop, freqs)``: the N × J genotype matrix, the population of
    each sample and the n_pops × J matrix of population allele frequencies.

    ``G`` is stored as int8 to save memory: convert with ``G.astype(float)`` before arithmetic
    whose intermediate results can exceed 127.  A rare SNP can come out monomorphic (no
    variation) in a small sample — skip such columns in per-SNP tests.
    """
    p = rng.uniform(*maf_range, n_snps)
    if n_pops > 1 and fst > 0:
        a, b = p * (1 - fst) / fst, (1 - p) * (1 - fst) / fst
        freqs = np.clip(rng.beta(a, b, size=(n_pops, n_snps)), 0.001, 0.999)
    else:
        freqs = np.tile(p, (n_pops, 1))
    pop = np.repeat(np.arange(n_pops), int(np.ceil(n_samples / n_pops)))[:n_samples]
    G = rng.binomial(2, freqs[pop]).astype(np.int8)
    return G, pop, freqs


def accuracy(y_true, y_pred) -> float:
    """Fraction of correct predictions."""
    return float(np.mean(np.asarray(y_true) == np.asarray(y_pred)))


print("🧰 Toolkit loaded:", ", ".join(['simulate_leukemia_expression', 'simulate_genotypes', 'accuracy']))

In [ ]:
# ── 0.1 Generate the leukaemia data (SIMULATED, fixed default seed) ──
leuk = simulate_leukemia_expression()
X_g, y_g, is_train = leuk["X"], leuk["y"], leuk["is_train"]
GENE_INDEX = {g: i for i, g in enumerate(leuk["genes"])}      # gene symbol → column
print("source:", leuk["source"], "|", X_g.shape[0], "patients ×", X_g.shape[1], "genes")
print("original split: training", is_train.sum(), f"({y_g[is_train].sum()} AML) | test", (~is_train).sum(),
      f"({y_g[~is_train].sum()} AML)")

---
## 1 · The learning problem, stated precisely

> 📍 **Where we are.** Module 00 described supervised and unsupervised learning in words; Module 02 gave you
> probability. Here we combine them, exactly as the first 02-620 lecture does: every learning problem becomes
> "estimate a probability distribution from data".

**Supervised learning** means learning to perform the task "given input $X$, decide output $Y$": given blood-test
results $X$, decide the diagnosis $Y$ (0 = healthy, 1 = heart disease); given an image $X$, decide whether it shows a
cat. It is learning *with a teacher*: the data are pairs $D = \{(\mathbf x^1, y^1), \dots, (\mathbf x^N, y^N)\}$ — an
input (expression profile, genotype) with the correct answer (diagnosis, phenotype). From these **training data** we
learn a function $f: X \to Y$ and then use it on previously unseen inputs, where nobody knows $Y$. If $Y$ is a
continuous value the task is **regression**; if $Y$ is discrete, **classification**. To model uncertainty, the problem
of learning $f$ is formulated as learning the conditional distribution

$$P(Y \mid X;\ \theta) .$$

*In words:* for each input $X$, the model gives a probability for every possible output, and the probabilities add up
to one. For an image, $P(\text{cat}\mid X) = 0.9$ and $P(\text{dog}\mid X) = 0.1$ means "fairly sure it is a cat", while
0.5/0.5 means "no idea" — the same bare answer "cat" hides very different certainty. $\theta$ (theta) stands for the
model's unknown **parameters**.

**Unsupervised learning** means "find interesting patterns in $X$": group images of animals into horses, cats, dogs;
group newspaper articles by topic; group patients or genes. It is learning *without a teacher*: the data are inputs
only, $D = \{\mathbf x^1, \dots, \mathbf x^N\}$. In probability terms it is formulated as learning the **density**

$$P(X;\ \theta) .$$

*In words:* learn how the inputs themselves are distributed — which profiles are common, which rare, which genes vary
together. It is much harder than supervised learning: nobody tells you what the right answer is, and even evaluating
the result is a question in itself (how do we know an unsupervised result is right? Modules 12–14 and 23 return to it).

| Example | Input $X$ | Output $Y$ | Type | Course module |
|---|---|---|---|---|
| Finding genes in a genome (methods borrowed from speech recognition) | DNA sequence | gene / non-gene label per position | supervised (sequence labelling) | HMMs, 19–20 |
| Genetic basis of complex diseases (cancer, diabetes, arthritis, heart disease) | genotypes (whole genome) | disease status or trait | supervised | regression/classification, 06–11 |
| Golub ALL vs AML: class prediction and class discovery | expression of ~7,000 genes | leukaemia type | supervised (and clustering) | 08–12 |
| Gene regulatory networks from microarrays of hundreds of individuals | expression profiles | — (the network is hidden) | unsupervised | graphical models, 15–18 |
| Population structure in Europe (197,146 loci, 1,387 people → 2 dimensions) | SNP genotypes | — | unsupervised | PCA 14, mixtures 13 |

The three kinds of learner side by side (an overview borrowed from Andrew Moore's machine-learning tutorials):

In [ ]:
# ── 1.0 Supervised vs unsupervised learning in one picture: three kinds of learner ──
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.set_xlim(0, 10); ax.set_ylim(0, 3.6); ax.axis("off")
rows_ = [(2.9, "Classifier", "prediction of a categorical\noutput (a class)", "supervised", "tab:blue"),
         (1.8, "Regressor", "prediction of a real-valued\noutput", "supervised", "tab:blue"),
         (0.7, "Density estimator", "probability of the input\nP(x)", "unsupervised", "tab:green")]
for yy, name, out, kind, col in rows_:
    ax.text(1.0, yy, "input\nattributes x", ha="center", va="center", fontsize=8,
            bbox=dict(boxstyle="round", fc="0.95", ec="0.6"))
    ax.annotate("", xy=(3.2, yy), xytext=(1.9, yy), arrowprops=dict(arrowstyle="-|>", color="0.4"))
    ax.text(4.3, yy, name, ha="center", va="center", fontsize=9, color="white", weight="bold",
            bbox=dict(boxstyle="round,pad=0.5", fc=col, ec=col))
    ax.annotate("", xy=(6.6, yy), xytext=(5.5, yy), arrowprops=dict(arrowstyle="-|>", color="0.4"))
    ax.text(7.9, yy, out, ha="center", va="center", fontsize=8)
    ax.text(9.6, yy, kind, ha="center", va="center", fontsize=8, color=col, style="italic")
ax.plot([0.2, 9.9], [1.25, 1.25], color="0.7", ls="--", lw=0.8)
ax.set_title("supervised learning models P(Y | X); unsupervised learning models P(X)")
plt.tight_layout(); plt.show()

In [ ]:
# ── 1.1 Two views of the same gene: P(X) without labels, P(Y | X) with labels (SIMULATED leukaemia data, CD33) ──
cd33 = X_g[:, GENE_INDEX["CD33"]]
edges = np.quantile(cd33, [1 / 3, 2 / 3])                 # split patients into low / middle / high thirds
bin_of = np.digitize(cd33, edges)
rows = []
for k, name in enumerate(["low", "middle", "high"]):
    m = bin_of == k
    rows.append(dict(CD33_level=name, range=f"{cd33[m].min():.2f}–{cd33[m].max():.2f}", patients=m.sum(),
                     AML=y_g[m].sum(), P_AML_given_level=y_g[m].mean()))
print(pd.DataFrame(rows).round(3).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].hist(cd33, bins=20, color="0.6", edgecolor="white")
axes[0].set_xlabel("CD33 expression (log10)"); axes[0].set_ylabel("number of patients")
axes[0].set_title("SIMULATED leukaemia data — unsupervised view:\nan estimate of P(X) (no labels used)")
p_bin = [r["P_AML_given_level"] for r in rows]
axes[1].bar(["low", "middle", "high"], p_bin, color=["tab:blue", "0.6", "tab:red"])
for i, p in enumerate(p_bin):
    axes[1].text(i, p + 0.02, f"{p:.2f}", ha="center")
axes[1].set_ylim(0, 1.1); axes[1].set_xlabel("CD33 expression level (thirds)"); axes[1].set_ylabel("P(AML | level)")
axes[1].set_title("supervised view: an estimate of P(Y = AML | X)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The left histogram uses the expression values alone: it estimates $P(X)$, the unsupervised
target. The right panel uses the labels too and estimates $P(Y = \text{AML}\mid X)$ by counting: among the 24 patients
with the lowest CD33 expression, none has AML; in the middle third 6 of 24 (0.25); in the highest third 19 of 24
(0.79). CD33 is a
myeloid surface marker, so this makes biological sense — and the output is a *probability*, which tells a clinician how
sure the prediction is. Notice that each of these numbers is itself an estimate (a fraction) computed from data: the
rest of this notebook is about how to estimate such quantities well.

✏️ **Your turn.** Repeat with five bins instead of three (`np.quantile(cd33, [0.2, 0.4, 0.6, 0.8])`). What happens to
the estimates in the extreme bins, and how many patients support each estimate? Keep this "fewer data per estimate"
problem in mind for Section 8.

### 📘 Lecture notes — supervised and unsupervised learning

**Supervised learning** — "given input $X$, decide output $Y$" — learns a function $f: X \to Y$ from training pairs
$(X, Y)$ in which $Y$ acts as the teacher, then applies $f$ to previously unseen inputs. Continuous $Y$:
**regression** (a *regressor* predicts a real value); discrete $Y$: **classification** (a *classifier* predicts a
class). To represent uncertainty the problem is formulated as learning $P(Y\mid X)$, with
$\sum_y P(Y = y\mid X) = 1$: $P(\text{cat}\mid X) = 0.9$, $P(\text{dog}\mid X) = 0.1$ (fairly sure) vs 0.5/0.5 (no idea).

**Unsupervised learning** — "find interesting patterns in $X$" (clusters of images, topics of documents, groups of
patients) — has inputs only, no teacher. It is formulated as learning the density $P(X)$ (a *density estimator*). It
is much harder than supervised learning, and evaluating its accuracy is itself a problem, since no labels exist.

| | supervised | unsupervised |
|---|---|---|
| data | input $X$ and output $Y$ | input $X$ only |
| model | $P(Y\mid X)$ | $P(X)$ |
| learning | with a teacher | without a teacher |

**"What is X, what is Y, what is f?"** Gene finding: X = DNA sequence, Y = gene/non-gene labels, f = a hidden Markov
model (supervised). Disease genetics: X = genotypes, Y = disease or trait, f = regression/classification. Golub's
leukaemias: X = expression profile, Y = ALL/AML (classification; clustering for class discovery). Gene networks: X =
expression profiles of hundreds of individuals; the "output" — the network — is unknown (unsupervised, graphical
models). Population structure: X = 197,146 genotypes per person, reduced to 2 dimensions by PCA (unsupervised).

**Worked example (Section 1.1).** CD33 in the simulated leukaemia data: a histogram of the 72 values estimates
$P(X)$; the fraction of AML patients within each third of CD33 expression estimates $P(Y = \text{AML}\mid X)$.

> 🎓 **Exam tip.** For any application be ready to name X, Y and whether the task is supervised (regression or classification) or
    unsupervised, and to say which distribution is modelled: $P(Y\mid X)$ or $P(X)$.

---
## 2 · The likelihood and the maximum-likelihood estimate

> 📍 **Where we are.** A model is a family of probability distributions indexed by $\theta$. "Solving" for a
> distribution means choosing a family (say, Gaussian) and then deciding its unknown parameters. Which member of the
> family should we pick? Maximum likelihood answers: the θ that makes the observed data most probable,
> $\theta_{MLE} = \arg\max_\theta P(D\mid\theta)$ — found, when the log-likelihood is differentiable, by
> differentiating $\log P(D\mid\theta)$ with respect to θ and setting it to zero (Module 04).

**The model.** Each isolate is resistant ($x = 1$) with probability $q$ and susceptible ($x = 0$) with probability
$1-q$: a **Bernoulli**($q$) random variable (Module 02), written compactly as

$$P(x\mid q) = q^{x}\,(1-q)^{1-x}, \qquad x \in\{0, 1\} .$$

*In words:* if $x = 1$ the formula gives $q^1(1-q)^0 = q$; if $x = 0$ it gives $1-q$ — one expression for both cases.

🧮 **Math refresher — independence, the product symbol and "arg max".** Two events are **independent** if the
probability that both happen is the product of their probabilities, $P(A \text{ and } B) = P(A)\,P(B)$ (Module 02). The
symbol $\prod_{i=1}^{N} a_i$ means $a_1 \times a_2 \times \dots \times a_N$ — repeated multiplication, just as $\sum$
means repeated addition. Finally, $\arg\max_\theta f(\theta)$ means "*the value of* $\theta$ at which $f$ is largest" —
the *location* of the peak, not its height ($\max_\theta f(\theta)$ would be the height).

**The likelihood.** If the isolates are **independent and identically distributed** (i.i.d.: each one an independent
draw from the same population), the probability of the whole data set is the product of the individual probabilities:

$$L(q) = P(D\mid q) = \prod_{i=1}^{N} q^{x_i}(1-q)^{1-x_i} = q^{\,n_R}\,(1-q)^{\,n_S} ,$$

*In words:* multiply $q$ once for every resistant isolate and $(1-q)$ once for every susceptible one; $n_R$ and $n_S$
are the two counts. Read as a function of $q$ with the data fixed, this is the **likelihood**: how probable the observed
data would be if the true frequency were $q$.

⚠️ **Misconception.** The likelihood is *not* the probability that $q$ is the true value. It is a probability *of the
data*; as a function of $q$ it need not integrate to 1. (The probability of $q$ given the data is the posterior of
Section 4.)

In [ ]:
# ── 2.1 The likelihood of R, R, S, S, R, R, S (= the coin flips H, H, T, T, H, H, T) ──
isolates = np.array([1, 1, 0, 0, 1, 1, 0])               # 1 = resistant (heads), 0 = susceptible (tails)
n_R, n_S = isolates.sum(), (1 - isolates).sum()

def likelihood(q, x=isolates):
    """L(q) = Π q^x (1 − q)^(1 − x) for Bernoulli data x."""
    return np.prod(q ** x * (1 - q) ** (1 - x))


for q in [0.2, 0.5, 4 / 7, 0.8]:
    terms = " × ".join(f"{t:.3f}" for t in q ** isolates * (1 - q) ** (1 - isolates))
    print(f"q = {q:.3f}: L = {terms} = {likelihood(q):.6f}")
qq = np.linspace(0, 1, 100001)
area = np.sum([likelihood(q) for q in qq[::100]]) * (qq[100] - qq[0])
print(f"area under L(q) from 0 to 1 ≈ {area:.5f}  (exactly 1/280 = {1 / 280:.5f}, not 1: a likelihood is not a density in q)")

🔍 **What just happened?** Each likelihood is a product of seven factors, one per isolate. $q = 0.5$ gives
$0.5^7 = 0.0078$; $q = 4/7$ gives a slightly larger value, 0.0084; $q = 0.2$ or $0.8$ make the data much less probable.
The area under $L(q)$ is $1/280$, confirming that the likelihood is not a probability distribution over $q$.

### 2.2 Maximising the log-likelihood

Take logs (Module 04: same maximiser, products become sums):

$$\ell(q) = \log L(q) = n_R\log q + n_S\log(1-q), \qquad
\frac{d\ell}{dq} = \frac{n_R}{q} - \frac{n_S}{1-q} = 0 \;\Longrightarrow\; \hat q_{MLE} = \frac{n_R}{n_R + n_S} = \frac{n_R}{N}.$$

*In words:* the slope of the log-likelihood balances the pull of the resistant isolates ($n_R/q$) against the
susceptible ones ($n_S/(1-q)$); they balance exactly when $q$ equals the observed fraction. For our data $\hat q = 4/7
\approx 0.571$. The second derivative $-n_R/q^2 - n_S/(1-q)^2$ is negative everywhere, so $\ell$ is concave and this is
the global maximum.

In [ ]:
# ── 2.2 MLE by calculus and by grid search ──
def loglik(q, nR, nS):
    """Bernoulli log-likelihood ℓ(q) = nR log q + nS log(1 − q)."""
    return nR * np.log(q) + nS * np.log(1 - q)


grid = np.linspace(0.001, 0.999, 9981)
q_grid = grid[np.argmax(loglik(grid, n_R, n_S))]
print(f"calculus: q̂ = nR / N = {n_R}/{n_R + n_S} = {n_R / (n_R + n_S):.4f} | grid search: {q_grid:.4f}")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].plot(grid, [likelihood(q) for q in grid], color="tab:blue"); axes[0].set_ylabel("likelihood L(q)")
axes[1].plot(grid, loglik(grid, n_R, n_S), color="tab:orange"); axes[1].set_ylabel("log-likelihood ℓ(q)"); axes[1].set_ylim(-20, 0)
for ax in axes:
    ax.axvline(4 / 7, color="k", ls="--", lw=1); ax.set_xlabel("resistance frequency q")
axes[0].set_title("L(q) = q⁴(1 − q)³: maximum at q = 4/7"); axes[1].set_title("ℓ(q): same maximiser, easier algebra")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The grid search finds the same maximiser as the formula, 0.5714. The likelihood and the
log-likelihood peak at the same $q$, but the log-likelihood is a smooth concave curve whose derivative is easy to write
— this is why 02-620 almost always maximises "$\log \prod$" rather than the product itself.

### 2.3 More data, sharper likelihood

In [ ]:
# ── 2.3 The same fraction 4/7 with 7, 70 and 700 isolates ──
fig, ax = plt.subplots(figsize=(6.4, 3.2))
for k, c in zip([1, 10, 100], ["tab:blue", "tab:orange", "tab:green"]):
    ll = loglik(grid, 4 * k, 3 * k)
    rel = np.exp(ll - ll.max())                            # likelihood divided by its maximum (avoids underflow)
    inside = grid[rel > np.exp(-2)]                        # q values within a factor e² of the best
    ax.plot(grid, rel, color=c, label=f"N = {7 * k}: plausible q ∈ [{inside.min():.2f}, {inside.max():.2f}]")
ax.axvline(4 / 7, color="k", ls="--", lw=1)
ax.set_xlabel("resistance frequency q"); ax.set_ylabel("L(q) / max L"); ax.legend(fontsize=7)
ax.set_title("same MLE (4/7), but more data → a narrower likelihood"); plt.tight_layout(); plt.show()

🔍 **What just happened?** All three data sets have the same MLE, but with 700 isolates the likelihood is a narrow
spike, while with 7 it is broad: many values of $q$ explain seven isolates almost equally well. (The legend reports the
range of $q$ whose likelihood is within a factor $e^2 \approx 7.4$ of the best — a rough "plausible range" that shrinks
roughly like $1/\sqrt N$.) The MLE alone does not tell you how certain it is; the Bayesian posterior of Section 4 does.

### 2.4 The population-genetics version: an allele frequency from genotypes

A diploid genotype $g \in \{0, 1, 2\}$ counts copies of allele T. Under Hardy–Weinberg equilibrium (Module 02) each of
the two copies is T with probability $p$, independently, so $g \sim \text{Binomial}(2, p)$:

$$P(g\mid p) = \binom{2}{g} p^{g}(1-p)^{2-g}, \qquad
\ell(p) = \text{const} + \Big(\sum_i g_i\Big)\log p + \Big(2N - \sum_i g_i\Big)\log(1-p) \;\Longrightarrow\; \hat p = \frac{\sum_i g_i}{2N} .$$

*In words:* the genotypes contribute $\sum_i g_i$ T alleles out of $2N$ allele copies, and the MLE is simply the
fraction of T alleles — **allele counting**. It is the coin-flip MLE with each person contributing two "flips"; the
binomial coefficient $\binom{2}{g}$ (1, 2, 1) does not depend on $p$ and drops out.

Without assuming Hardy–Weinberg, the three genotype frequencies $\pi_0, \pi_1, \pi_2$ form a **categorical** model
(a "multinoulli", as the naïve-Bayes lecture calls it) with the constraint $\pi_0 + \pi_1 + \pi_2 = 1$. Maximising
$\sum_k n_k\log\pi_k$ with a Lagrange multiplier (Module 04, Section 7) gives

$$\frac{n_k}{\pi_k} = \lambda \;\;\text{for every } k \;\Longrightarrow\; \hat\pi_k = \frac{n_k}{N} .$$

*In words:* the MLE of a categorical distribution is the observed fraction in each category — the count ratios that
train a naïve-Bayes classifier (Section 2.5 and Module 08).

In [ ]:
# ── 2.4 Allele-frequency MLE: formula vs grid; genotype frequencies vs Hardy–Weinberg (SIMULATED) ──
rng = np.random.default_rng(SEED + 2)
G, _, freqs = simulate_genotypes(rng, n_samples=40, n_snps=1, maf_range=(0.3, 0.3))
g = G[:, 0].astype(int)
p_hat = g.sum() / (2 * len(g))
ll_p = g.sum() * np.log(grid) + (2 * len(g) - g.sum()) * np.log(1 - grid)
print(f"true p = {freqs[0, 0]:.2f} | allele counting p̂ = {g.sum()}/{2 * len(g)} = {p_hat:.4f} | grid search {grid[ll_p.argmax()]:.4f}")
n_k = np.bincount(g, minlength=3)
hwe = np.array([(1 - p_hat) ** 2, 2 * p_hat * (1 - p_hat), p_hat ** 2])
print(pd.DataFrame({"genotype": ["0 (AA)", "1 (AT)", "2 (TT)"], "count": n_k, "MLE n_k/N": n_k / len(g),
                    "HWE from p̂": hwe}).round(3).to_string(index=False))

🔍 **What just happened?** For 40 simulated people the allele-counting formula and the grid search give the same
$\hat p = 24/80 = 0.30$ — here it happens to equal the true value exactly, which is luck: rerun with another seed and it
will typically be off by a few hundredths. The categorical MLE (observed genotype fractions 0.45/0.50/0.05) and the
Hardy–Weinberg predictions from $\hat p$ (0.49/0.42/0.09) are similar but not identical: two different models of the
same data, one with 2 free parameters and one with 1, and 40 people are too few to tell them apart.

✏️ **Your turn.** Rerun with `n_samples=4` several times (change the seed). How often is $\hat p$ off by more than 0.15?
How often is one genotype class completely missing, so that its categorical MLE is exactly 0?

### 2.5 From one coin to a whole classifier: conditional vs joint likelihood

The same counting principle trains an entire classifier. Two flavours of likelihood appear in 02-620:

* **Conditional likelihood** $\prod_{l=1}^{L} P(Y^l\mid X^l)$ — maximised by linear and logistic regression (the maximum
  conditional likelihood estimate, MCLE): the model describes only how $Y$ depends on $X$.
* **Joint likelihood** $\prod_{l=1}^{L} P(Y^l, X^l)$ — maximised by **naïve Bayes**, which models the whole joint
  distribution $P(X, Y) = P(Y)\,P(X\mid Y)$ and assumes the features are conditionally independent given the class,
  $P(X\mid Y) = \prod_{i=1}^{n} P(X_i\mid Y)$.

For $L$ training samples $\langle X^l, Y^l\rangle$ with $n$ features each, the naïve-Bayes log-likelihood falls apart
into independent pieces:

$$\log\prod_{l=1}^{L} P(Y^l)\prod_{i=1}^{n}P(X_i^l\mid Y^l)
= \underbrace{\sum_{l}\log P(Y^l)}_{\text{only } \pi}
+ \underbrace{\sum_{l}\log P(X_1^l\mid Y^l)}_{\text{only the table of feature 1}} + \dots + \sum_{l}\log P(X_n^l\mid Y^l) .$$

*In words:* the log turns the product into a sum, and each term involves its own parameters only, so each can be
maximised on its own. The first term is a coin-flip log-likelihood for $\pi = P(Y = 1)$, so $\hat\pi$ = fraction of
samples with $Y = 1$. Each other term is a categorical log-likelihood inside each class, so

$$\hat\theta_{ijk} = \hat P(X_i = k\mid Y = j) = \frac{\#(\text{samples with } X_i = k \text{ and } Y = j)}{\#(\text{samples with } Y = j)} .$$

*In words:* maximum likelihood for naïve Bayes is **counting** — the count ratios of Section 2.4, computed separately in
each class. We check this on the six-individual genotype table of the naïve-Bayes lecture (loci $X_1, X_2, X_3$ coded
0 = AA, 1 = AT, 2 = TT; $Y = 1$ cancer, $Y = 0$ healthy).

In [ ]:
# ── 2.5 Naïve-Bayes MLEs are count ratios: the six-individual table of the naïve-Bayes lecture ──
nb = pd.DataFrame(NB_GENOTYPE_TABLE, columns=["X1", "X2", "X3", "Y"])
nb.index = [f"person {i}" for i in range(1, 7)]
print(nb.to_string())
Xnb, ynb = nb[["X1", "X2", "X3"]].to_numpy(), nb["Y"].to_numpy()
pi_nb = ynb.mean()                                                  # π̂ = #(Y = 1) / L
theta_nb = np.array([[[np.mean(Xnb[ynb == j, i] == k) for k in range(3)] for j in range(2)] for i in range(3)])
print(f"\nπ̂ = P(Y = 1) = {ynb.sum()}/{len(ynb)} = {pi_nb:.3f}")
print("θ̂ for locus 1 — rows: class Y = j, columns: genotype X1 = k (count ratios within each class):")
print(pd.DataFrame(theta_nb[0], index=["Y = 0 (healthy)", "Y = 1 (cancer)"], columns=["X1=0", "X1=1", "X1=2"]).round(3).to_string())

def nb_joint_loglik(pi, theta):
    """Σ_l log P(Y^l)  and the list [Σ_l log P(X_i^l | Y^l) for each locus i] (the separate pieces)."""
    ll_y = np.sum(np.log(np.where(ynb == 1, pi, 1 - pi)))
    return ll_y, [np.sum(np.log(theta[i, ynb, Xnb[:, i]])) for i in range(3)]


ll_y, ll_x = nb_joint_loglik(pi_nb, theta_nb)
print(f"\njoint log-likelihood at the count ratios = {ll_y:.3f} (P(Y) piece) + "
      + " + ".join(f"{v:.3f}" for v in ll_x) + f" (one piece per locus) = {ll_y + sum(ll_x):.3f}")
rng = np.random.default_rng(SEED + 25)
better = 0
for _ in range(2000):                                               # any other parameters do worse
    p_try, th_try = rng.uniform(0.02, 0.98), rng.dirichlet(np.ones(3), size=(3, 2))
    a_, b_ = nb_joint_loglik(p_try, th_try)
    better += (a_ + sum(b_)) > ll_y + sum(ll_x)
print(f"2,000 random parameter sets: {better} reach a higher joint log-likelihood than the count ratios")

🔍 **What just happened?** Counting gives $\hat\pi = 3/6 = 0.5$ and, for locus 1, $\hat P(X_1 = k\mid Y = 1) = (2/3, 1/3, 0)$
(cancer patients 1–3 have genotypes 0, 1, 0) and $\hat P(X_1 = k\mid Y = 0) = (0, 1/3, 2/3)$ (healthy persons 4–6 have 2,
2, 1). The joint log-likelihood at these count ratios is the sum of four separate pieces, and none of 2,000 random
parameter sets does better — the count ratios are the maximum. Two of the six locus-1 estimates are exactly 0; Section 4.3
explains why such zeros are dangerous and how a prior repairs them.

### 📘 Lecture notes — maximum likelihood: from the coin flip to naïve Bayes

**The principle.** For a probability model with parameter θ, write down the likelihood — the probability of the
observed data given the model — and choose the θ that maximises it: $\theta_{MLE} = \arg\max_\theta P(D\mid\theta)$.
If the log-likelihood is differentiable, set $\partial\log P(D\mid\theta)/\partial\theta = 0$.

**The coin-flip example.** $X \sim$ Bernoulli($q$), $q$ = probability of heads; data H, H, T, T, H, H, T. For i.i.d.
flips $L(q) = q^4(1-q)^3$, $\ell(q) = 4\log q + 3\log(1-q)$, $\ell'(q) = 4/q - 3/(1-q) = 0$, so

$$\hat q_{MLE} = \frac{n_H}{n_H + n_T} = \frac47 \approx 0.571 .$$

*In words:* the MLE of a probability is the observed fraction. For $K$ categories (a "multinoulli"), a Lagrange
multiplier for $\sum_k\pi_k = 1$ gives $\hat\pi_k = n_k/N$; for diploid genotypes under Hardy–Weinberg, allele
counting $\hat p = \sum_i g_i/2N$.

**Conditional vs joint likelihood.** Regression maximises $\prod_l P(Y^l\mid X^l)$ (MCLE); naïve Bayes maximises
$\prod_l P(Y^l, X^l) = \prod_l P(Y^l)\prod_i P(X_i^l\mid Y^l)$ (MLE of the joint). Its log separates into
$\sum_l\log P(Y^l) + \sum_i\sum_l\log P(X_i^l\mid Y^l)$, so $\hat\pi$ and every table are estimated separately:
$\hat\theta_{ijk} = \#(X_i = k, Y = j)/\#(Y = j)$.

**Worked example (Section 2.5).** Six individuals, three loci: $\hat\pi = 3/6$; $\hat P(X_1 = 1\mid Y = 1) = 1/3$;
$\hat P(X_1 = 0\mid Y = 0) = 0/3 = 0$.

**Common confusions.** The likelihood is a probability of the *data*, not of θ (its area over $q$ is $1/280$ here,
not 1). The i.i.d. assumption is what turns the joint probability of the data into a product.

> 🎓 **Exam tip.** Be able to derive $\hat q = n_H/N$ in three lines (likelihood → log → derivative = 0) and to generalise it to
    $K$ categories ($\hat\pi_k = n_k/N$) and to naïve Bayes (count ratios within each class). A common slip is to forget
    the i.i.d. assumption that turns the joint probability into a product.

---
## 3 · Maximum likelihood for a Gaussian: mean and variance of a gene

> 📍 **Where we are.** Coins and genotypes are discrete. Expression levels, blood pressure and cholesterol are
> continuous and often modelled as Gaussian — the noise model of linear regression (Module 06) and of Gaussian naïve
> Bayes (Module 08). Same recipe, two parameters.

🧮 **Math refresher — the Gaussian density** (Module 02). A Gaussian (normal) random variable with mean $\mu$ and
variance $\sigma^2$ has density

$$p(x\mid\mu,\sigma^2) = \frac{1}{\sqrt{2\pi\sigma^2}}\exp\!\Big(-\frac{(x-\mu)^2}{2\sigma^2}\Big) .$$

*In words:* a bell curve centred at $\mu$ whose width is set by $\sigma$; values far from $\mu$ (in units of $\sigma$)
are exponentially unlikely.

**Log-likelihood of $N$ i.i.d. values** $x_1, \dots, x_N$:

$$\ell(\mu, \sigma^2) = -\frac N2\log(2\pi\sigma^2) - \frac{1}{2\sigma^2}\sum_{i=1}^{N}(x_i - \mu)^2 .$$

*In words:* the first term rewards small variances, the second penalises squared distances from $\mu$ (scaled by the
variance). Setting both partial derivatives to zero:

$$\frac{\partial\ell}{\partial\mu} = \frac{1}{\sigma^2}\sum_i (x_i - \mu) = 0 \Rightarrow \hat\mu = \frac1N\sum_i x_i, \qquad
\frac{\partial\ell}{\partial\sigma^2} = -\frac{N}{2\sigma^2} + \frac{1}{2\sigma^4}\sum_i (x_i-\mu)^2 = 0 \Rightarrow
\hat\sigma^2 = \frac1N\sum_i (x_i - \hat\mu)^2 .$$

*In words:* the MLE of the mean is the sample mean, and the MLE of the variance is the average squared deviation —
dividing by $N$. Note that maximising $\ell$ over $\mu$ is the same as **minimising the squared error** $\sum_i (x_i -
\mu)^2$ (Module 04, Section 2): least squares *is* Gaussian maximum likelihood, which is exactly how the regression
lecture justifies linear regression — with $\mu$ replaced by $\mathbf x^\top\mathbf w$ for each sample (Module 06).

In [ ]:
# ── 3.1 Gaussian MLE for TCF3 in ALL and AML patients (SIMULATED): formulas vs numerical optimisation ──
tcf3 = X_g[:, GENE_INDEX["TCF3"]]

def neg_loglik_gauss(params, x):
    """−ℓ(μ, σ²) with σ parameterised as exp(log σ) so the optimiser cannot make it negative."""
    mu, log_sigma = params
    return -np.sum(stats.norm.logpdf(x, mu, np.exp(log_sigma)))


fits = {}
for c, name in [(0, "ALL"), (1, "AML")]:
    x = tcf3[y_g == c]
    mu_hat, var_hat = x.mean(), np.mean((x - x.mean()) ** 2)          # the formulas
    res = minimize(neg_loglik_gauss, x0=[2.0, 0.0], args=(x,))        # brute-force maximisation
    fits[name] = (mu_hat, np.sqrt(var_hat))
    print(f"{name} (N = {len(x)}): μ̂ = {mu_hat:.4f}, σ̂² = {var_hat:.5f} | optimiser: μ = {res.x[0]:.4f}, "
          f"σ² = {np.exp(2 * res.x[1]):.5f} | np.var = {np.var(x):.5f}, np.var(ddof=1) = {np.var(x, ddof=1):.5f}")

xs = np.linspace(1.9, 3.8, 300)
fig, ax = plt.subplots(figsize=(6.4, 3.3))
for (name, (m, s)), c in zip(fits.items(), ["tab:blue", "tab:red"]):
    ax.hist(tcf3[leuk["labels"] == name], bins=12, density=True, alpha=0.35, color=c, label=f"{name} patients")
    ax.plot(xs, stats.norm.pdf(xs, m, s), color=c, lw=2, label=rf"Gaussian MLE: $\hat\mu$ = {m:.2f}, $\hat\sigma$ = {s:.2f}")
ax.set_xlabel("TCF3 expression (log10)"); ax.set_ylabel("density"); ax.legend(fontsize=7)
ax.set_title("SIMULATED leukaemia data: one Gaussian per type, fitted by maximum likelihood"); plt.tight_layout(); plt.show()

🔍 **What just happened?** For both leukaemia types the closed-form MLEs agree with a general-purpose numerical
optimiser to four decimals, and with `np.var` (which divides by $N$ by default). TCF3, a lymphoid transcription
factor, is higher in ALL ($\hat\mu = 3.01$) than in AML ($\hat\mu = 2.27$), and each class is reasonably described by a
bell curve ($\hat\sigma = 0.14$ and $0.16$).

🧪 **Ground-truth check.** The simulator built TCF3 with a true AML − ALL mean difference of −0.80 log10 units; the two
MLEs give $2.266 - 3.011 = -0.75$. Close, but not exact: 47 and 25 patients carry sampling noise, and the detection floor
at 2.0 compresses the lowest AML values upward. With real data we could never make this comparison. Two class-specific Gaussians plus the class frequencies are a complete **generative model** of
(expression, diagnosis) — Section 9 turns it into a classifier, and Module 08 calls it Gaussian naïve Bayes.

⚠️ **Pitfall: $N$ or $N-1$?** `np.var(x)` divides by $N$ (the MLE) but `pandas.Series.var()` and `np.var(x, ddof=1)`
divide by $N - 1$. Both are legitimate estimators; know which one your software uses.

### 3.2 The MLE of a variance is biased

In [ ]:
# ── 3.2 On average, the 1/N variance underestimates σ² by the factor (N − 1)/N ──
rng = np.random.default_rng(SEED + 32)
print(" N   mean of σ̂²_MLE (1/N)   (N−1)/N   mean of s² (1/(N−1))     [true σ² = 1, 20,000 data sets each]")
for N_ in [2, 3, 5, 10, 50]:
    data = rng.normal(0.0, 1.0, size=(20000, N_))
    v_mle = data.var(axis=1)                       # ddof = 0
    v_unb = data.var(axis=1, ddof=1)
    print(f"{N_:2d}   {v_mle.mean():20.3f}   {(N_ - 1) / N_:7.3f}   {v_unb.mean():20.3f}")

🔍 **What just happened?** Averaged over 20,000 simulated data sets, the MLE variance comes out at about
$(N-1)/N$ times the truth (0.5 for $N=2$, 0.8 for $N=5$, 0.98 for $N=50$): it is **biased** low. The reason: $\hat\mu$
is fitted to the same data, so the data are on average closer to $\hat\mu$ than to the true $\mu$. Dividing by $N-1$
removes the bias. The difference vanishes as $N$ grows; the lesson that *estimators fitted and evaluated on the same
data look too good* returns in a much stronger form as overfitting (Section 9, Modules 06–07 and 23).

✏️ **Your turn.** The MLE of the variance has a smaller **variance** than the unbiased estimator. Compute
`v_mle.var()` and `v_unb.var()` for $N = 5$, and then the mean squared error `np.mean((v - 1)**2)` of each. Which
estimator has the smaller MSE? (Section 5 explains why bias is not the whole story.)

---
## 4 · Bayes' rule for parameters: the MAP estimate

> 📍 **Where we are.** MLE uses the data only — the frequentist approach, often called "objective". The first 02-620
> lecture introduces the alternative: the Bayesian approach, maximum a posteriori (MAP) estimation, which estimates
> parameters based on the observed data **and your belief** (and is criticised for being "subjective"). Module 02 taught
> Bayes' rule for events; here we apply it to the unknown parameter itself.

🧮 **Math refresher — Bayes' rule in machine learning.** $D$ is the data (the evidence); $\theta$ the unknown
quantities (model parameters, predictions). Treat the unknown $\theta$ as a random
variable with a **prior** distribution $P(\theta)$ that expresses what we believe before seeing data $D$. Then

$$P(\theta\mid D) = \frac{P(D\mid\theta)\,P(\theta)}{P(D)}, \qquad P(D) = \int P(D\mid\theta)\,P(\theta)\,d\theta .$$

*In words:* **posterior** (belief after seeing the data) = **likelihood** (how probable the data are under $\theta$)
× **prior** (belief before the data), divided by the **evidence** $P(D)$, which makes the posterior integrate to 1. (The
integral $\int \dots\, d\theta$ is the continuous version of a sum $\sum$ over all possible values of $\theta$ — the area
under the curve; for a discrete $\theta$ it *is* a sum, exactly as in the Bayes' rule of Module 02.) The evidence does not
depend on $\theta$, so it does not change *where* the posterior is largest. Hence
$P(D\mid\theta)P(\theta) = P(\theta\mid D)P(D) \propto P(\theta\mid D)$, where "$\propto$" (often written "$\sim$" in
lecture notes) means "is proportional to, as a function of $\theta$", and MAP estimation is defined as

$$\theta_{MAP} = \arg\max_\theta\, P(D\mid\theta)\,P(\theta) = \arg\max_\theta\,\big[\log P(D\mid\theta) + \log P(\theta)\big] .$$

*In words:* the MAP estimate maximises likelihood × prior — the log-likelihood plus a "log-prior" term that pulls the
estimate toward values the prior finds plausible.

🧮 **Math refresher — the Beta distribution**, the standard prior for a probability $q \in [0,1]$:

$$\text{Beta}(q\mid a, b) = \frac{q^{a-1}(1-q)^{b-1}}{B(a, b)}, \qquad \text{mean} = \frac{a}{a+b}, \qquad \text{mode} = \frac{a-1}{a+b-2}\ \ (a, b > 1) .$$

*In words:* a flexible bump on $[0,1]$; $a$ and $b$ act like earlier counts of successes and failures ($B(a,b)$ is just
the constant that makes the area 1). Beta(1, 1) is flat (uniform); Beta(3, 7) is a bump around 0.25–0.3; larger $a+b$
means a more confident prior.

In [ ]:
# ── 4.1 Prior × likelihood → posterior on a grid (resistance data; prior from earlier surveillance ≈ Beta(3, 7)) ──
a0, b0 = 3.0, 7.0                                          # prior belief: resistance around 25–30 %
prior = stats.beta.pdf(grid, a0, b0)
lik = np.exp(loglik(grid, n_R, n_S))
post_grid = prior * lik
post_grid /= post_grid.sum() * (grid[1] - grid[0])         # divide by P(D) so the area is 1
post_exact = stats.beta.pdf(grid, a0 + n_R, b0 + n_S)      # conjugacy: Beta(a + nR, b + nS)
print(f"max |grid posterior − Beta({a0 + n_R:.0f}, {b0 + n_S:.0f}) density| = {np.abs(post_grid - post_exact).max():.2e}")
print(f"MLE = {n_R / (n_R + n_S):.3f} | MAP (grid) = {grid[post_grid.argmax()]:.3f} | "
      f"MAP formula (nR + a − 1)/(N + a + b − 2) = {(n_R + a0 - 1) / (n_R + n_S + a0 + b0 - 2):.3f} | "
      f"posterior mean (a + nR)/(a + b + N) = {(a0 + n_R) / (a0 + b0 + n_R + n_S):.3f}")

fig, ax = plt.subplots(figsize=(6.4, 3.3))
ax.plot(grid, prior, color="tab:green", label=f"prior Beta({a0:.0f}, {b0:.0f})")
ax.plot(grid, lik / (lik.sum() * (grid[1] - grid[0])), color="tab:blue", label="likelihood q⁴(1−q)³ (rescaled)")
ax.plot(grid, post_grid, color="tab:red", lw=2, label=f"posterior Beta({a0 + n_R:.0f}, {b0 + n_S:.0f})")
for v, c in [(4 / 7, "tab:blue"), ((n_R + a0 - 1) / (n_R + n_S + a0 + b0 - 2), "tab:red")]:
    ax.axvline(v, color=c, ls="--", lw=1)
ax.set_xlabel("resistance frequency q"); ax.set_ylabel("density"); ax.legend(fontsize=7)
ax.set_title("SIMULATED isolates R,R,S,S,R,R,S: the posterior compromises"); plt.tight_layout(); plt.show()

🔍 **What just happened?** Multiplying the prior (green) by the likelihood (blue) and rescaling gives the posterior
(red), which sits between them: the data say 0.57, the prior says about 0.25, and the posterior peaks at **0.40**. The
grid posterior matches the Beta(7, 10) density to numerical precision — the result of **conjugacy**:

$$P(q\mid D) \;\propto\; \underbrace{q^{n_R}(1-q)^{n_S}}_{\text{likelihood}}\;\underbrace{q^{a-1}(1-q)^{b-1}}_{\text{Beta prior}}
= q^{\,n_R + a - 1}(1-q)^{\,n_S + b - 1} \;\Longrightarrow\; q\mid D \sim \text{Beta}(a + n_R,\ b + n_S) .$$

*In words:* with a Beta prior, the posterior is again a Beta distribution whose parameters are the prior parameters plus
the observed counts — updating a belief is just adding counts. Maximising its log, $(n_R + a - 1)\log q + (n_S + b -
1)\log(1-q)$, exactly as in Section 2.2, gives

$$\hat q_{MAP} = \frac{n_R + a - 1}{N + a + b - 2} .$$

*In words:* the MAP estimate is the MLE computed after adding $a-1$ **imaginary** resistant isolates and $b-1$
imaginary susceptible ones. With $a = 3$, $b = 7$: $(4 + 2)/(7 + 8) = 6/15 = 0.40$. The posterior *mean*,
$(a + n_R)/(a + b + N) = 7/17 = 0.41$, is a slightly different summary of the same posterior.

In [ ]:
# ── 4.2 Different priors, different MAP estimates (same 7 isolates) ──
def q_map(nR, nS, a, b):
    """MAP estimate of a Bernoulli probability under a Beta(a, b) prior."""
    return (nR + a - 1) / (nR + nS + a + b - 2)


rows = []
for a, b, story in [(1, 1, "flat prior (no opinion)"), (2, 2, "one imaginary R and one imaginary S"),
                    (3, 7, "surveillance: resistance ≈ 25 %"), (20, 20, "strong belief in 50 %")]:
    post = stats.beta.pdf(grid, a + n_R, b + n_S)
    rows.append(dict(prior=f"Beta({a}, {b})", meaning=story, imaginary_R=a - 1, imaginary_S=b - 1,
                     MAP_formula=q_map(n_R, n_S, a, b), MAP_grid=grid[post.argmax()]))
print(pd.DataFrame(rows).round(3).to_string(index=False))
print(f"MLE for comparison: {n_R / (n_R + n_S):.3f}")

🔍 **What just happened?** A flat Beta(1, 1) prior adds no imaginary isolates, so **MAP = MLE** (0.571). Beta(2, 2) adds
one of each and nudges the estimate toward 0.5 (5/9 = 0.556). The surveillance prior pulls it to 0.40, and a strong
Beta(20, 20) prior — worth 38 imaginary isolates, versus 7 real ones — holds it near 0.5 (23/45 = 0.511). The prior's
influence is set by its **strength** $a + b - 2$ relative to the sample size $N$.

### 4.3 The zero-count problem — in the naïve-Bayes table

Now the scenario of the biological question: the first three isolates are all susceptible.

In [ ]:
# ── 4.3 Zero counts: three susceptible isolates, and the six-individual genotype table of Section 2.5 ──
print(f"3 isolates S, S, S: MLE = {0 / 3:.3f} | MAP with Beta(2, 2) = {q_map(0, 3, 2, 2):.3f} | "
      f"posterior mean = {(0 + 2) / (3 + 4):.3f}")
print("\nlocus 1 of the naïve-Bayes training table (genotypes 0 = AA, 1 = AT, 2 = TT; Y = 1 cancer):")
rows = []
for yv in [1, 0]:
    x1 = nb.loc[nb.Y == yv, "X1"].to_numpy()
    for k in range(3):
        n_k, n_y = int(np.sum(x1 == k)), len(x1)
        rows.append(dict(parameter=f"P(X1={k} | Y={yv})", count=f"{n_k}/{n_y}", MLE=n_k / n_y,
                         MAP_one_imaginary_each=(n_k + 1) / (n_y + 3)))     # Dirichlet(2, 2, 2): +1 per genotype
print(pd.DataFrame(rows).round(3).to_string(index=False))

🔍 **What just happened?** With three susceptible isolates, the MLE declares resistance **impossible** (0.000), while the
MAP estimate with one imaginary isolate of each kind gives 0.2 and the posterior mean $2/7 \approx 0.29$ — "rare but
possible", the answer an expert would give.

The naïve-Bayes table of Section 2.5 has exactly this problem. Among the three healthy individuals ($Y = 0$) the
locus-1 genotypes are 2, 2, 1, so the MLE of $P(X_1 = 0\mid Y = 0)$ is $0/3 = 0$; the MLE of $P(X_1 = 1\mid Y = 1)$ is
1/3, and $P(X_1 = 2\mid Y = 1) = 0/3$ is a second zero. A new person with genotype AA at locus 1 would therefore get probability
exactly zero of being healthy, *whatever* their other genotypes: a single zero factor wipes out the whole naïve-Bayes
product $P(Y)\prod_i P(X_i\mid Y)$, no matter how strongly the other loci point to "healthy". This is the naïve-Bayes
lecture's "subtlety #1": if unlucky, an MLE of $P(X_i\mid Y)$ is zero simply because no training sample happened to
have that value in that class. Adding one imaginary individual per
genotype value (a Dirichlet prior, the multi-category version of the Beta) gives $(0+1)/(3+3) = 1/6$ instead. For $K$
categories with prior parameters $\alpha_k$:

$$\hat\pi_k^{MAP} = \frac{n_k + \alpha_k - 1}{N + \sum_{k'}\alpha_{k'} - K} .$$

*In words:* add $\alpha_k - 1$ imaginary observations to category $k$ and renormalise; with every $\alpha_k = 2$ this is
the classic "add-one" (Laplace) smoothing.

✏️ **Your turn.** Compute the MLE and the add-one MAP estimates of $P(X_2 = k\mid Y = 0)$ and $P(X_3 = k\mid Y = 1)$ for
$k = 0, 1, 2$ from the table. How many of the 18 class-conditional MLE parameters are exactly zero?

### 📘 Lecture notes — MLE vs MAP, and Bayes' rule in machine learning

**Two principles for solving for a distribution's parameters.** *Frequentist:* maximum likelihood estimation (MLE)
— estimate the parameters from the observed data only ("an objective approach"). *Bayesian:* maximum a posteriori
(MAP) estimation — estimate them from the observed data **and your belief** (criticised for being "subjective").
Both are used throughout the course, for supervised and unsupervised learning alike.

**Bayes' rule in machine learning.** With $D$ the data (evidence) and $\theta$ the unknown quantities (model
parameters, predictions):

$$P(\theta\mid D) = \frac{P(D\mid\theta)\times P(\theta)}{P(D)} .$$

*In words:* posterior (belief about θ *after* seeing $D$) = likelihood (how likely the observed data are under
this θ) × prior (belief *before* seeing $D$) ÷ evidence. Learning the posterior means updating the prior with the
data.

**MAP.** Because $P(D\mid\theta)P(\theta) = P(\theta\mid D)P(D) \propto P(\theta\mid D)$ (the evidence does not depend on θ),

$$\theta_{MAP} = \arg\max_\theta P(\theta\mid D) = \arg\max_\theta P(D\mid\theta)P(\theta),$$

*In words:* the MAP estimate is the most probable parameter value after seeing the data; since the evidence $P(D)$
is the same for every θ, it is found by maximising likelihood × prior — like the MLE: take the log,
$\log P(D\mid\theta) + \log P(\theta)$, differentiate, set to 0. The only difference from MLE is the extra log-prior term.

**Worked example (Section 4.1).** Isolates R,R,S,S,R,R,S with a Beta(3, 7) prior: posterior Beta(3 + 4, 7 + 3) =
Beta(7, 10); $\hat q_{MAP} = (4 + 2)/(7 + 8) = 0.40$ vs $\hat q_{MLE} = 4/7 = 0.571$; posterior mean $7/17 = 0.41$.
With a flat Beta(1, 1) prior MAP = MLE.

> 🎓 **Exam tip.** Know why $P(D)$ can be ignored for MAP but not for computing posterior probabilities, and that with a flat
    prior MAP = MLE. Don't confuse the posterior *mode* (MAP) with the posterior *mean*.

### 📘 Lecture notes — zero counts and "imaginary examples" (the naïve-Bayes lecture)

**The problem.** If unlucky, an MLE of $P(X_i\mid Y)$ is exactly zero: no training sample has that feature value in
that class. Why worry about one parameter out of many? Because the naïve-Bayes score $P(Y)\prod_i P(X_i\mid Y)$ is a
product — if one factor is 0, the whole class probability is 0, whatever the other features say.

**MLE vs MAP for naïve Bayes.** MLE: choose θ that maximises the probability of the observed data,
$\arg\max_\theta P(D\mid\theta)$. MAP: choose the θ that is most probable given the prior and the data,
$\arg\max_\theta P(\theta\mid D) = \arg\max_\theta P(D\mid\theta)P(\theta)/P(D)$. With Beta (two values) or Dirichlet
(several values) priors the estimates become

$$\hat\pi_{MLE} = \frac{\#(Y = 1)}{L}, \quad \hat\pi_{MAP} = \frac{\#(Y = 1) + \alpha_0}{L + \alpha_0 + \beta_0};\qquad
\hat\theta_{ijk}^{MLE} = \frac{\#(X_i = k, Y = j)}{\#(Y = j)}, \quad \hat\theta_{ijk}^{MAP} = \frac{\#(X_i = k, Y = j) + m_k}{\#(Y = j) + \sum_{k'} m_{k'}} .$$

*In words:* the only difference is **"imaginary" examples** — $\alpha_0$ imaginary samples with $Y = 1$, $\beta_0$ with
$Y = 0$, $m_k$ with $X_i = k$ in each class — added to the real counts (for a Dirichlet prior with parameters $\alpha_k$, $m_k = \alpha_k - 1$, as in Section 4.3). In the notation of a Beta($a, b$) prior,
$\alpha_0 = a - 1$ and $\beta_0 = b - 1$ (the formula is the posterior mode), so $\alpha_0 = \beta_0 = 1$ is Beta(2, 2):
add-one (Laplace) smoothing.

**Worked example (Section 4.3).** Healthy individuals have locus-1 genotypes 2, 2, 1: $\hat P(X_1 = 0\mid Y = 0) = 0/3$
(MLE) but $(0 + 1)/(3 + 3) = 1/6$ with one imaginary individual per genotype.

> 🎓 **Exam tip.** If an exam gives a Beta($a, b$) prior, the MAP is $(n_1 + a - 1)/(N + a + b - 2)$, not $(n_1 + a)/(N + a + b)$ —
    the latter is the posterior *mean*. When a formula is written with pseudo-counts $\alpha_0, \beta_0$ directly, they
    are the *imaginary examples* ($a - 1$, $b - 1$), not the Beta parameters.

---
## 5 · MLE vs MAP as data accumulate

> 📍 **Where we are.** MAP needs a prior; MLE does not. Which estimator is *better*? We answer by simulation: repeat the
> same study 20,000 times on a population with known resistance frequency $q = 0.3$ and look at the spread of each
> estimator around the truth. (This is the frequentist way of judging any estimator, including Bayesian ones.)

For an estimator $\hat q$ (a random quantity, because the data are random):

$$\text{bias} = \mathbb E[\hat q] - q, \qquad \text{variance} = \mathbb E\big[(\hat q - \mathbb E[\hat q])^2\big], \qquad
\text{MSE} = \mathbb E\big[(\hat q - q)^2\big] = \text{bias}^2 + \text{variance} .$$

*In words:* the **bias** is how far the estimator is off *on average*; the **variance** is how much it jumps around
from study to study; the **mean squared error** combines both. A slightly biased estimator with much smaller variance can
have a smaller MSE — the **bias–variance trade-off** at the heart of regularisation (Module 07) and model selection
(Module 23).

In [ ]:
# ── 5.1 20,000 replicate studies per sample size: bias, variance and MSE of four estimators ──
rng = np.random.default_rng(SEED + 5)
q_true, R = 0.3, 20000
Ns = np.array([3, 5, 10, 20, 50, 100, 300, 1000])
estimators = {"MLE": (1, 1), "MAP Beta(2,2)": (2, 2), "MAP Beta(3,7) (good prior)": (3, 7),
              "MAP Beta(30,30) (strong, wrong prior)": (30, 30)}
results = {name: {"bias": [], "var": [], "mse": []} for name in estimators}
for N_ in Ns:
    nR = rng.binomial(N_, q_true, size=R)                  # number of resistant isolates in each replicate study
    for name, (a, b) in estimators.items():
        est = q_map(nR, N_ - nR, a, b)
        results[name]["bias"].append(est.mean() - q_true)
        results[name]["var"].append(est.var())
        results[name]["mse"].append(np.mean((est - q_true) ** 2))
    if N_ == 3:
        print(f"N = 3: fraction of studies with MLE exactly 0 = {np.mean(nR == 0):.3f} (theory 0.7³ = {0.7 ** 3:.3f})")
tab = pd.DataFrame({name: r["mse"] for name, r in results.items()}, index=Ns).T
print("\nmean squared error (rows = estimator, columns = N):"); print(tab.to_string(float_format=lambda v: f"{v:.4f}"))
r = results["MAP Beta(2,2)"]
print(f"\ncheck MSE = bias² + variance for MAP Beta(2,2), N = 3: {r['mse'][0]:.5f} = {r['bias'][0] ** 2 + r['var'][0]:.5f}")

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.4))
for (name, r), c in zip(results.items(), ["k", "tab:blue", "tab:green", "tab:red"]):
    axes[0].loglog(Ns, r["mse"], "o-", ms=3, color=c, label=name)
    axes[1].semilogx(Ns, r["bias"], "o-", ms=3, color=c, label=name)
axes[0].set_xlabel("number of isolates N"); axes[0].set_ylabel("mean squared error"); axes[0].legend(fontsize=7)
axes[0].set_title("SIMULATED: MSE of each estimator (true q = 0.3)")
axes[1].axhline(0, color="0.6", lw=0.8); axes[1].set_xlabel("number of isolates N"); axes[1].set_ylabel("bias")
axes[1].set_title("the prior's bias fades as N grows"); plt.tight_layout(); plt.show()

🔍 **What just happened?** Read the MSE table column by column.

* **Small samples (N = 3).** The MLE is unbiased but wildly variable — about a third of all three-isolate studies
  (34.8 % in the simulation; theory $0.7^3 = 34.3$ %) report a frequency of exactly zero — and has the largest MSE
  (0.071). Every MAP estimator beats it, even the one
  with a strong prior centred on the wrong value (0.5), because shrinking toward *any* reasonable value reduces variance
  a lot. The prior that is roughly right (Beta(3, 7)) is best by far (0.007).
* **Large samples (N ≥ 100).** The data overwhelm a weak prior: MLE, Beta(2, 2) and Beta(3, 7) become nearly
  indistinguishable (0.0018–0.0021 at N = 100, identical to four decimals from N = 300), and their MSE falls like $1/N$.
  The strong wrong prior is now clearly *worse* than the MLE (0.0062 vs 0.0021 at N = 100): its bias (right panel)
  shrinks only slowly because 58 imaginary isolates take many real ones to outweigh.
* The identity MSE = bias² + variance holds exactly for the simulated estimates.

So MAP is not "better" or "worse" than MLE in general: a prior helps when data are scarce and costs a bias that fades as
data accumulate — at a rate set by the prior's strength. In 02-620 the prior's strength is a dial ($\sigma_0^2$ in ridge
regression, $\lambda$ in logistic regression, pseudo-counts in naïve Bayes) that is usually tuned on held-out data
(Module 23).

✏️ **Your turn.** Add the estimator "MAP Beta(4, 10)" (also centred near 0.3 but stronger) and the "posterior mean under
Beta(2, 2)", $(n_R + 2)/(N + 4)$. Where do they fall in the table?

---
## 6 · A Gaussian prior on a mean shrinks the estimate

> 📍 **Where we are.** Section 5 used a Beta prior for a probability. For a continuous parameter such as a mean expression
> change, the natural prior is a Gaussian. The result is the formula behind ridge regression (Module 07) — seen here
> with a single parameter first, then for a whole regression.

🧫 You measure the log fold change of a gene between treated and control cells in **three** replicates, with known
measurement noise $\sigma$. Model: $x_i \sim \mathcal N(\mu, \sigma^2)$. Prior: most genes barely change, so
$\mu \sim \mathcal N(0, \sigma_0^2)$ with a smallish $\sigma_0$. The log-posterior is

$$\log P(\mu\mid D) = \text{const} - \frac{1}{2\sigma^2}\sum_{i=1}^{N}(x_i - \mu)^2 - \frac{1}{2\sigma_0^2}(\mu - 0)^2 .$$

*In words:* the data pull $\mu$ toward the sample mean, the prior pulls it toward 0; each pull is weighted by a
precision ($1/\text{variance}$). Setting the derivative to zero (Module 04):

$$\hat\mu_{MAP} = \frac{N/\sigma^2}{N/\sigma^2 + 1/\sigma_0^2}\;\bar x \qquad\Longleftrightarrow\qquad
\hat\mu_{MAP} = \arg\min_\mu\; \sum_i (x_i - \mu)^2 + \frac{\sigma^2}{\sigma_0^2}\,\mu^2 .$$

*In words:* the MAP estimate is the sample mean multiplied by a **shrinkage factor** between 0 and 1. Equivalently, it
minimises squared error plus a penalty $\lambda\mu^2$ with $\lambda = \sigma^2/\sigma_0^2$ — exactly the ridge objective
$(\mathbf y - X\mathbf w)^\top(\mathbf y - X\mathbf w) + (\sigma^2/\sigma_0^2)\,\mathbf w^\top\mathbf w$ of the regression
lecture. A small $\sigma_0^2$ (strong belief that effects are small) means a large $\lambda$ and strong shrinkage.

**The same for a whole regression (the regression lecture's MAP estimate).** When $N < J$ the least-squares estimate
does not exist (Module 03, Section 5), so the lecture takes the Bayesian route. Bayes' rule for the weights reads
posterior ∝ likelihood × prior, $P(\mathbf w\mid\mathbf y, X) \propto P(\mathbf y\mid X, \mathbf w)\,P(\mathbf w)$, with the
Gaussian likelihood of linear regression and a Gaussian prior on every regression coefficient,

$$\mathbf w \sim \mathcal N\big(\mathbf 0, \sigma_0^2 I\big) \quad (\text{no prior on the intercept } w_0), \qquad
\log P(\mathbf w) = \text{const} - \frac{1}{2\sigma_0^2}\mathbf w^\top\mathbf w .$$

*In words:* each weight is believed to be near 0 (no preference for positive or negative effects), independently of
the others (covariance $\sigma_0^2 I$, zero off the diagonal); a lower $\sigma_0$ means a stronger pull toward 0. Adding
the log-prior to the log-likelihood and multiplying by $-2\sigma^2$:

$$\mathbf w_{MAP} = \arg\min_{\mathbf w}\Big[(\mathbf y - X\mathbf w)^\top(\mathbf y - X\mathbf w) + \frac{\sigma^2}{\sigma_0^2}\mathbf w^\top\mathbf w\Big]
= \Big(X^\top X + \frac{\sigma^2}{\sigma_0^2}I\Big)^{-1}X^\top\mathbf y .$$

*In words:* least squares plus a penalty on large weights; the solution (derived in Module 04, Section 6.4) adds the
"ridge" $\sigma^2/\sigma_0^2$ to the diagonal of $X^\top X$, which makes the matrix invertible even if $N < J$. From the
frequentist side this is "add a small number to the diagonal"; from the Bayesian side it is a prior with variance
$\sigma_0^2$ — the same estimator.

In [ ]:
# ── 6.1 Shrinkage path for one gene, and the average error over 2,000 SIMULATED genes ──
SIGMA = 1.0                                                 # known replicate noise (log2 units)
x3 = np.array([1.2, 0.4, 0.9])                              # three replicate log fold changes of one gene

def mu_map(xbar, N, sigma, sigma0, mu0=0.0):
    """MAP estimate of a Gaussian mean with prior N(mu0, sigma0²): precision-weighted average of x̄ and mu0."""
    w_data, w_prior = N / sigma ** 2, 1 / sigma0 ** 2
    return (w_data * xbar + w_prior * mu0) / (w_data + w_prior)


s0_grid = np.logspace(-1.5, 1.5, 60)
path = mu_map(x3.mean(), 3, SIGMA, s0_grid)
for s0 in [0.1, 0.3, 1.0, 3.0, 10.0]:
    print(f"σ0 = {s0:5.1f} (λ = σ²/σ0² = {SIGMA ** 2 / s0 ** 2:7.2f}): μ̂_MAP = {mu_map(x3.mean(), 3, SIGMA, s0):.3f}   (MLE x̄ = {x3.mean():.3f})")

rng = np.random.default_rng(SEED + 6)
TAU = 0.5                                                   # true effects are mostly small: μ_g ~ N(0, 0.5²)
mu_true = rng.normal(0, TAU, 2000)
xbar_g = rng.normal(mu_true[:, None], SIGMA, size=(2000, 3)).mean(axis=1)
mse_mle = np.mean((xbar_g - mu_true) ** 2)
mse_map = np.array([np.mean((mu_map(xbar_g, 3, SIGMA, s0) - mu_true) ** 2) for s0 in s0_grid])
print(f"\n2,000 genes: MSE of the MLE = {mse_mle:.3f} (theory σ²/N = {SIGMA ** 2 / 3:.3f}); best MAP MSE = {mse_map.min():.3f} "
      f"at σ0 = {s0_grid[mse_map.argmin()]:.2f} (true spread of effects τ = {TAU})")

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.3))
axes[0].semilogx(s0_grid, path, color="tab:purple"); axes[0].axhline(x3.mean(), color="k", ls="--", lw=1, label="MLE x̄")
axes[0].axhline(0, color="0.6", lw=0.8); axes[0].set_xlabel("prior SD σ0 (log scale)"); axes[0].set_ylabel(r"$\hat\mu_{MAP}$")
axes[0].set_title("one gene: a stronger prior (small σ0) shrinks toward 0"); axes[0].legend(fontsize=7)
axes[1].semilogx(s0_grid, mse_map, color="tab:purple", label="MAP"); axes[1].axhline(mse_mle, color="k", ls="--", lw=1, label="MLE")
axes[1].axvline(TAU, color="tab:green", ls=":", lw=1, label="σ0 = true τ")
axes[1].set_xlabel("prior SD σ0 (log scale)"); axes[1].set_ylabel("MSE over 2,000 genes")
axes[1].set_title("SIMULATED genes: moderate shrinkage helps most"); axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** For the single gene, the MAP estimate moves from the sample mean 0.833 (weak prior, large
$\sigma_0$) down toward 0 (strong prior): with $\sigma_0 = 1$, for example, the three replicates and the prior get weights
3 and 1, so $\hat\mu = 0.75 \times 0.833 = 0.625$. Across 2,000 simulated genes whose true effects are mostly small, the
MLE's average squared error is 0.34 (theory $\sigma^2/N = 0.33$), while MAP with a well-chosen $\sigma_0$ cuts it by more
than half (0.15); the best $\sigma_0$ (0.47) lands close to the true spread of effects $\tau = 0.5$. Too strong a prior (very small $\sigma_0$)
shrinks real effects away; too weak a prior gives back the MLE. Shrinkage estimators built on this idea are widely used to
stabilise fold-change estimates from few replicates, and the same trade-off decides the ridge penalty in Module 07.

✏️ **Your turn.** Make the true effects sparse instead: `mu_true = np.where(rng.random(2000) < 0.9, 0, rng.normal(0, 2, 2000))`
(90 % of genes unchanged, 10 % with large effects). Does MAP with a Gaussian prior still beat the MLE? Which genes does
it hurt? (Module 07 shows that a Laplace prior — the lasso — suits sparse effects better.)

---
## 7 · Frequentist vs Bayesian: two philosophies, three kinds of answer

> 📍 **Where we are.** The introduction lecture contrasts MLE ("objective") and MAP ("criticised for being
> subjective"). Both give a single number. The full Bayesian approach (Module 21) keeps the whole posterior.

| | Frequentist (MLE) | Bayesian, MAP | Fully Bayesian |
|---|---|---|---|
| What is uncertain? | the data (θ is fixed but unknown) | θ itself | θ itself |
| Uses a prior? | no | yes | yes |
| Answer | $\hat\theta = \arg\max P(D\mid\theta)$ | $\arg\max P(D\mid\theta)P(\theta)$ | the whole $P(\theta\mid D)$ |
| Uncertainty statement | confidence interval (over repeated studies) | (none by itself) | credible interval: $P(\theta\in[a,b]\mid D) = 0.95$ |
| Prediction for a new case | plug in $\hat\theta$ | plug in $\hat\theta_{MAP}$ | average over the posterior |
| Where in 02-620 | least squares, naïve-Bayes counts, logistic regression | ridge, lasso, smoothed naïve Bayes, regularised LR | MCMC, Module 21 |

The "subjectivity" criticism is real but often overstated: the prior is stated explicitly (and can be checked), its
influence fades with data (Section 5), and many "objective" choices — which model, which features — are subjective too.

In [ ]:
# ── 7.1 Three answers for the seven isolates (prior Beta(3, 7)) ──
post = stats.beta(a0 + n_R, b0 + n_S)                       # Beta(7, 10)
q_mle = n_R / (n_R + n_S)
se = np.sqrt(q_mle * (1 - q_mle) / (n_R + n_S))             # textbook standard error of a proportion
print(f"MLE                       : {q_mle:.3f}, approx. 95 % confidence interval [{q_mle - 1.96 * se:.3f}, {q_mle + 1.96 * se:.3f}]")
print(f"MAP                       : {q_map(n_R, n_S, a0, b0):.3f}")
print(f"posterior mean            : {post.mean():.3f}, 95 % credible interval [{post.ppf(0.025):.3f}, {post.ppf(0.975):.3f}]")
print(f"P(q > 0.5 | data)         : {post.sf(0.5):.3f}   ← a direct probability statement about the parameter")
print(f"P(next isolate resistant) : plug-in MLE {q_mle:.3f} | Bayesian predictive (= posterior mean) {post.mean():.3f}")

🔍 **What just happened?** With only seven isolates the textbook 95 % confidence interval around the MLE is very wide
(roughly 0.20–0.94) and the approximation behind it is poor for small $N$. The Bayesian posterior gives a 95 % credible
interval (roughly 0.20–0.65) and answers questions the MLE cannot, such as "how probable is it that resistance exceeds
50 %?" (about 0.23). The predictive probability that the next isolate is resistant is the posterior mean, 0.41. All of
these Bayesian numbers depend on the prior — change `a0, b0` and they move, which is both the strength and the weakness
behind the "subjective" criticism.

✏️ **Your turn.** Recompute the three answers with a flat prior Beta(1, 1) and with Beta(30, 70) (the same prior mean
0.3, but ten times stronger). Which conclusions — "resistance is probably below 50 %", "the next isolate is more
likely susceptible" — survive all three priors? Reporting such a **sensitivity analysis** is good Bayesian practice.

---
## 8 · Learning a joint distribution, then conditioning

> 📍 **Where we are.** The probabilistic approach to supervised learning models $f: X \to Y$ with $P(Y\mid X)$. The
> introduction lecture shows an alternative route with a classic teaching example (from Andrew Moore's tutorials on
> probability, built from US census records): to learn the function $f: \langle G, H\rangle \to W$ that predicts wealth
> $W$ (poor/rich) from gender $G$ and weekly hours worked $H$ — equivalently $P(W\mid G, H)$ — **learn the joint
> distribution $P(W, G, H)$ from data, then calculate $P(W\mid G, H)$**. Once you have the joint distribution, you can
> ask for the probability of any logical expression involving your attributes. Nobody expects such a model to predict
> wealth well; it is an illustration. We work through its numbers, apply the idea to a biological cohort, and find
> its fatal weakness.

🧮 **Math refresher — conditioning a joint table** (Module 02):

$$P(W\mid G, H) = \frac{P(W, G, H)}{P(G, H)} = \frac{P(W, G, H)}{\sum_{w} P(W = w, G, H)} .$$

*In words:* keep the rows of the joint table that match the condition, and renormalise them so they add up to one.

In [ ]:
# ── 8.1 The classic joint table P(G, H, W): gender G, hours worked per week H, wealth W (census records) ──
joint = pd.DataFrame({
    "G": ["Female"] * 4 + ["Male"] * 4,
    "H": ["<40.5", "<40.5", "≥40.5", "≥40.5"] * 2,
    "W": ["poor", "rich"] * 4,
    "P": [0.253122, 0.0245895, 0.0421768, 0.0116293, 0.331313, 0.0971295, 0.134106, 0.105933]})
print(f"sum of the eight probabilities = {joint.P.sum():.7f} (1 up to rounding)")
cond = joint.pivot_table(index=["G", "H"], columns="W", values="P")
cond["P(rich | G, H)"] = cond["rich"] / (cond["rich"] + cond["poor"])
print(cond.round(4).to_string())
print(f"\nmarginal P(rich) = {joint.loc[joint.W == 'rich', 'P'].sum():.4f};  "
      f"P(rich | Female) = {joint.query('W == \"rich\" and G == \"Female\"').P.sum() / joint.query('G == \"Female\"').P.sum():.4f}")

🔍 **What just happened?** From the eight joint probabilities we built the whole functional mapping
$\langle G, H\rangle \to P(W\mid G, H)$ — the exercise the lecture leaves to the students. For example, among women
working 40.5 hours or more, $P(\text{rich}) = 0.0116/(0.0422 + 0.0116) = 0.216$; among men working that much, 0.441. We
also answered questions the "function" was never trained for, such as the marginal $P(\text{rich}) = 0.24$ or
$P(\text{rich}\mid\text{Female}) = 0.11$ — the power of having the joint.

### 8.2 A biological cohort: genotype, smoking and disease (SIMULATED)

In [ ]:
# ── 8.2 Estimate P(G, S, D) by counting, then condition to get P(D = 1 | G, S) ──
rng = np.random.default_rng(SEED + 8)

def simulate_cohort(rng, n):
    """SIMULATED cohort: risk-allele count G (HWE, p = 0.3), smoking S (25 %), disease D from a logistic model."""
    G = rng.binomial(2, 0.3, n); S = (rng.random(n) < 0.25).astype(int)
    p_d = 1 / (1 + np.exp(-(-3.0 + 0.7 * G + 1.2 * S)))     # true P(D = 1 | G, S)
    return pd.DataFrame({"G": G, "S": S, "D": (rng.random(n) < p_d).astype(int)})


def conditional_table(df, pseudo=0.0):
    """P(D = 1 | G, S) from counts, with `pseudo` imaginary cases of each outcome (MAP smoothing)."""
    out = []
    for G_ in range(3):
        for S_ in range(2):
            cell = df[(df.G == G_) & (df.S == S_)]
            n1, n = cell.D.sum(), len(cell)
            est = (n1 + pseudo) / (n + 2 * pseudo) if n + 2 * pseudo > 0 else np.nan
            out.append(dict(G=G_, S=S_, n=n, cases=n1, estimate=est, truth=1 / (1 + np.exp(-(-3 + 0.7 * G_ + 1.2 * S_)))))
    return pd.DataFrame(out)


big = simulate_cohort(rng, 2000)
joint_hat = big.value_counts(["G", "S", "D"], normalize=True).sort_index()     # 12-cell joint table, MLE = fractions
print("estimated joint P(G, S, D) (first 6 of 12 cells):", joint_hat.head(6).round(4).to_dict())
print("\nN = 2,000: P(D = 1 | G, S) by conditioning the counted joint")
print(conditional_table(big).round(3).to_string(index=False))
small = simulate_cohort(rng, 60)
print("\nN = 60: MLE vs MAP with one imaginary case and one imaginary control per cell")
t_small = conditional_table(small)
t_small["MAP (+1/+1)"] = conditional_table(small, pseudo=1.0)["estimate"]
print(t_small.round(3).to_string(index=False))

🔍 **What just happened?** With 2,000 simulated people, counting the 12 cells of the joint table and conditioning
recovers the true disease probabilities reasonably well — except in the rarest cell (two risk alleles and smoking), which
holds only 28 people, so its estimate (0.29) is far from the truth (0.40). With 60 people, several cells contain only a
handful of individuals (here 1, 3 and 5), and the MLEs become extreme: both cells with two risk alleles get an estimated
disease risk of exactly 0, although the true risks are 0.17 and 0.40. (An empty cell would give 0/0 — no estimate at
all.) The MAP estimates with one imaginary case and one imaginary control per cell (Section 4) are always defined and
less extreme. The joint
table works, but each cell needs data.

### 8.3 Why we cannot simply learn the joint: the exponential blow-up

In [ ]:
# ── 8.3 Free parameters of a full joint / conditional table vs naïve Bayes, and how many cells data can fill ──
n_snps = np.arange(1, 31)
full_joint = 2 * 3.0 ** n_snps - 1                 # joint over n ternary SNPs and a binary disease
full_cond = 3.0 ** n_snps                          # P(D = 1 | X1..Xn): one number per genotype combination
naive_bayes = 2 * n_snps * 2 + 1                   # naïve Bayes: 2 classes × n SNPs × 2 free numbers + 1 for P(Y)
print(f"n = 30 SNPs: full conditional table needs 3^30 = {3.0 ** 30:.2e} numbers; naïve Bayes needs {2 * 30 * 2 + 1}")

rng = np.random.default_rng(SEED + 83)
for n in [5, 10, 15]:
    Gc, _, _ = simulate_genotypes(rng, n_samples=2000, n_snps=n)
    seen = len({tuple(r) for r in Gc})
    print(f"n = {n:2d} SNPs: {3 ** n:>10,} genotype combinations; 2,000 people show {seen:5d} of them "
          f"({seen / 3 ** n:.2%}); every unseen combination has no data at all")

fig, ax = plt.subplots(figsize=(6.4, 3.3))
ax.semilogy(n_snps, full_joint, label="full joint P(X1..Xn, D): 2·3ⁿ − 1")
ax.semilogy(n_snps, full_cond, "--", label="full conditional P(D | X1..Xn): 3ⁿ")
ax.semilogy(n_snps, naive_bayes, label="naïve Bayes: 2·n·2 + 1")
ax.axhline(2000, color="0.5", ls=":", lw=1); ax.text(18, 5000, "a cohort of 2,000 people", fontsize=7, color="0.4")
ax.set_xlabel("number of SNPs n"); ax.set_ylabel("free parameters (log scale)"); ax.legend(fontsize=7)
ax.set_title("joint tables explode; conditional independence tames them"); plt.tight_layout(); plt.show()

🔍 **What just happened?** (The naïve-Bayes lecture calls the number of SNPs $n$; elsewhere in the course it is $J$.)
A full table over $n$ SNPs needs on the order of $3^n$ numbers — $2\times10^{14}$ for 30
SNPs, the number quoted in the naïve-Bayes lecture — while naïve Bayes needs $2\cdot n\cdot 2 + 1 = 121$. The simulation shows what this
means in practice: with 10 SNPs, 2,000 people display only a few percent of the 59,049 possible genotype combinations,
so most cells of the joint table would be estimated from **zero** observations; with 15 SNPs the observed fraction is a
tiny sliver. Learning the joint by counting is only possible for a handful of variables.

This is the second recurring idea of the course: **conditional independence buys tractability**. Naïve Bayes (Module 08)
assumes the SNPs are independent given the disease; Bayesian networks (Modules 15–17) factorise the joint along a graph
(31 → 10 parameters for the alarm network); hidden Markov models (Modules 19–20) assume a chain. Each model is a structured
joint distribution with far fewer parameters, estimated by exactly the MLE/MAP counting of Sections 2 and 4.

✏️ **Your turn.** For 20 binary genes, how many free parameters does the full joint table have? How many if all genes are
independent? How many if each gene depends on at most two "regulator" genes (each node's table then has $2^2 = 4$ rows)?

### 📘 Lecture notes — learning a joint distribution, then conditioning

**The idea.** Supervised learning models $f: X \to Y$ with $P(Y\mid X)$. One way to get it: learn the **joint**
distribution of all variables from data, then condition. To learn $f: \langle G, H\rangle \to W$ (wealth from gender
and hours worked), equivalently $P(W\mid G, H)$, estimate $P(W, G, H)$ and compute

$$P(W\mid G, H) = \frac{P(W, G, H)}{\sum_{w} P(W = w, G, H)} .$$

*In words:* keep the rows of the joint table that match the condition and renormalise them. Once you have the
joint, you can ask for the probability of any logical expression involving the attributes (marginals such as
$P(\text{rich})$, other conditionals such as $P(\text{Female}\mid\text{rich})$).

**Worked example (Section 8.1).** From the eight census probabilities: $P(\text{rich}\mid\text{Female}, \ge 40.5\text{ h}) =
0.0116/(0.0422 + 0.0116) = 0.216$; $P(\text{rich}\mid\text{Male}, \ge 40.5\text{ h}) = 0.441$; $P(\text{rich}) = 0.24$.

**Learning it by counting.** The MLE of each cell of a joint table is its observed fraction (Section 2.4); conditioning
the counted table gives cell-wise fractions of cases.

**Why it fails beyond a few variables.** A joint over $n$ variables with $k$ values each has $k^n - 1$ free
parameters; $P(Y\mid X_1, \dots, X_n)$ for a binary $Y$ needs one number per combination of the $X$'s ($3^n$ for $n$
SNPs: $3^{30} \approx 2\times10^{14}$). Most combinations are never observed, so their estimates rest on zero data.
**Conditional independence** rescues us: naïve Bayes needs $2\cdot n\cdot 2 + 1$ parameters for $n$ ternary SNPs and a
binary class (121 for 30 SNPs); Bayesian networks and HMMs factorise the joint along a graph.

> 🎓 **Exam tip.** Be able to condition a small joint table by hand and to count free parameters: a joint over $n$ variables with
    $k$ values each has $k^n - 1$; $P(Y\mid X_1..X_n)$ for binary $Y$ needs one number per combination of the $X$'s.

---
## 9 · Generalisation and the two-step recipe

> 📍 **Where we are.** Everything so far estimated parameters from data. But the point of learning is to do well on
> **new** data: supervised learning learns a function on training data and then uses it on previously unseen inputs,
> where we can compare its predictions with the truth. We finish by running the whole recipe on the simulated leukaemia
> data with the original study's split: 38 training patients to learn, 34 test patients to evaluate.

**Step 1 — choose a probability model.** A generative model for one gene $x$ and the diagnosis $y$:
$P(y) = \text{Bernoulli}(\pi)$ and $P(x\mid y = c) = \mathcal N(\mu_c, \sigma_c^2)$.
**Step 2 — estimate it by MLE** (Sections 2 and 3) on the training patients only. **Then predict** with Bayes' rule:

$$P(y = 1\mid x) = \frac{\pi\,\mathcal N(x;\mu_1,\sigma_1^2)}{\pi\,\mathcal N(x;\mu_1,\sigma_1^2) + (1-\pi)\,\mathcal N(x;\mu_0,\sigma_0^2)} .$$

*In words:* how well does each class's bell curve explain this patient's expression, weighted by how common the class is?
This is a one-gene Gaussian naïve-Bayes classifier (Module 08). We pick the gene using **training patients only**
(largest standardised difference between the class means, ignoring genes with values at the detection floor, which a
Gaussian cannot describe) — choosing it with the test patients would leak information and inflate the test score.

In [ ]:
# ── 9.1 Train a one-gene generative classifier on the 38 training patients, test on the other 34 ──
X_tr, y_tr, X_te, y_te = X_g[is_train], y_g[is_train], X_g[~is_train], y_g[~is_train]
m1, m0 = X_tr[y_tr == 1].mean(0), X_tr[y_tr == 0].mean(0)
sd_pool = np.sqrt((X_tr[y_tr == 1].var(0) * y_tr.sum() + X_tr[y_tr == 0].var(0) * (1 - y_tr).sum()) / len(y_tr))
usable = (X_tr > 2.0001).all(axis=0)                           # no values at the detection floor (in training data)
score = np.where(usable, np.abs(m1 - m0) / np.where(sd_pool > 0, sd_pool, np.inf), -np.inf)
j = int(np.argmax(score))
gene_j = leuk["genes"][j]
print(f"gene chosen on training data: {gene_j} (standardised mean difference {score[j]:.2f})")
print(f"🧪 the simulator's truth for {gene_j}: AML − ALL mean difference = {leuk['true_effect'][j]:+.2f} log10 units"
      + (f" ({LEUKEMIA_MARKERS[gene_j][1]}, a marker of {LEUKEMIA_MARKERS[gene_j][0]})" if gene_j in LEUKEMIA_MARKERS else ""))

pi_hat = y_tr.mean()                                           # Step 2: MLEs from the training patients
mu_c = [X_tr[y_tr == c, j].mean() for c in (0, 1)]
sd_c = [X_tr[y_tr == c, j].std() for c in (0, 1)]              # ddof = 0: the Gaussian MLE
print(f"π̂ = P(AML) = {y_tr.sum()}/{len(y_tr)} = {pi_hat:.3f};  ALL: μ̂ = {mu_c[0]:.3f}, σ̂ = {sd_c[0]:.3f};  "
      f"AML: μ̂ = {mu_c[1]:.3f}, σ̂ = {sd_c[1]:.3f}")

def p_aml(x):
    """P(AML | x) from the fitted generative model, via Bayes' rule (computed with logs)."""
    l1 = np.log(pi_hat) + stats.norm.logpdf(x, mu_c[1], sd_c[1])
    l0 = np.log(1 - pi_hat) + stats.norm.logpdf(x, mu_c[0], sd_c[0])
    return 1 / (1 + np.exp(l0 - l1))


for name, Xs, ys in [("training (38)", X_tr, y_tr), ("test (34)", X_te, y_te)]:
    p = p_aml(Xs[:, j])
    ll = np.mean(ys * np.log(p) + (1 - ys) * np.log(1 - p))
    print(f"{name:14s}: accuracy = {accuracy(ys, (p > 0.5).astype(int)):.3f}   average log P(true label | x) = {ll:.3f}")
wrong = (p_aml(X_te[:, j]) > 0.5) != y_te
print("🧪 misclassified test patients (sample_id):", leuk["sample_id"][~is_train][wrong].tolist(),
      "| flagged atypical by the simulator:", leuk["atypical"][~is_train][wrong].tolist())

In [ ]:
# ── 9.2 Picture of the fitted model and the test patients ──
xs = np.linspace(X_g[:, j].min() - 0.1, X_g[:, j].max() + 0.1, 300)
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.3))
for c, col, lab in [(0, "tab:blue", "ALL"), (1, "tab:red", "AML")]:
    prior_c = pi_hat if c == 1 else 1 - pi_hat
    axes[0].plot(xs, prior_c * stats.norm.pdf(xs, mu_c[c], sd_c[c]), color=col, label=rf"P(y = {lab}) · N(x; $\hat\mu$, $\hat\sigma^2$)")
    axes[0].plot(X_tr[y_tr == c, j], np.full((y_tr == c).sum(), -0.15), "|", color=col, ms=10)
axes[0].set_xlabel(f"{gene_j} expression (log10)"); axes[0].set_ylabel("weighted density")
axes[0].set_title("SIMULATED leukaemia data — Steps 1–2:\nclass models fitted on 38 training patients"); axes[0].legend(fontsize=7)
axes[1].plot(xs, p_aml(xs), color="k", label="P(AML | x) by Bayes' rule")
for c, col, lab in [(0, "tab:blue", "test ALL"), (1, "tab:red", "test AML")]:
    m = y_te == c
    axes[1].scatter(X_te[m, j], p_aml(X_te[m, j]), color=col, s=20, label=lab, zorder=3)
axes[1].axhline(0.5, color="0.6", ls=":", lw=1); axes[1].set_xlabel(f"{gene_j} expression (log10)")
axes[1].set_ylabel("P(AML | x)"); axes[1].set_title("prediction on 34 unseen test patients"); axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The gene picked from the training patients alone is CST3 (cystatin C), higher in AML — one
of the genes that distinguished the two leukaemias in the original study, and a true marker in the simulator (true
difference +0.86 log10 units, so the selection found a real signal). With five MLE parameters ($\pi$ and a mean and
standard deviation per class) the model classifies all 38 training patients correctly (100 %, with an average
log-probability of the true label of essentially 0 — total confidence) but only 32 of the 34 test patients (94 %), and
the average log-probability on the test set drops to −0.95. The two errors are AML patients with ALL-like CST3 levels
(3.59 and 3.63, against 3.91–4.16 for the other test AML patients); the model, whose class spreads were estimated from
only 11 and 27 training patients, gives them $P(\text{AML}) < 10^{-6}$ — confidently wrong. 🧪 These two (sample_id 47 and 67) are exactly
the two patients the simulator flags as *atypical* (a weak AML signature in all marker genes): no single-gene threshold could
have classified them. The training score is **optimistic**: the parameters — and the choice of gene — were tuned to
those very patients. This gap between training
and test performance is **generalisation error**, the central concern of Modules 06–07 (overfitting when $N < J$),
10 (trees), 22 (networks) and 23 (cross-validation). A single gene is also simply not enough: Golub *et al.* combined
50 genes, and Modules 08–11 show how to use many.

✏️ **Your turn.** Cheat on purpose: choose the gene with the *largest standardised difference computed on all 72
patients* (training + test), refit on the training patients and recompute the test accuracy. Then use the second- and
third-best training-set genes instead. Why is the "cheating" test accuracy not an honest estimate of how the classifier
would perform on new patients? (Ambroise & McLachlan (2002) showed how much this *selection bias* inflated published
accuracies; Module 23 returns to it.)

### The two-step recipe of 02-620

Every probabilistic method in the course is an instance of what you just did. The regression lecture states it as
its statistical principle — to learn $f: X \to Y$ from training data $\{\langle\mathbf x^1, y^1\rangle, \dots, \langle\mathbf x^N, y^N\rangle\}$,
"Step 1: choose some parameterized form for $P(Y\mid X;\theta)$ ($\theta$ the vector of parameters); Step 2: derive the
learning algorithm as the MLE or MAP estimate for $\theta$":

| Module | Step 1 — the probability model | Step 2 — MLE | Step 2 — MAP / prior |
|---|---|---|---|
| 06–07 linear regression | $P(y\mid\mathbf x) = \mathcal N(\mathbf x^\top\mathbf w, \sigma^2)$ | least squares $(X^\top X)^{-1}X^\top\mathbf y$ | Gaussian prior → ridge; Laplace prior → lasso |
| 08 naïve Bayes | $P(Y)\prod_i P(X_i\mid Y)$ | count ratios | Beta/Dirichlet pseudo-counts |
| 09 logistic regression | $P(Y = 1\mid\mathbf x) = \sigma(w_0 + \mathbf w^\top\mathbf x)$ | gradient ascent (concave) | Gaussian prior → L2 penalty |
| 13 mixture models | $P(\mathbf x) = \sum_k \pi_k\mathcal N(\mathbf x;\boldsymbol\mu_k,\Sigma_k)$ | EM (latent labels) | priors/variance floors |
| 15–18 graphical models | $\prod_i P(X_i\mid \text{parents})$ / Gaussian with sparse precision | counts / covariance | pseudo-counts / graphical lasso |
| 19–20 HMMs | Markov chain of hidden states + emissions | Baum–Welch (EM) | pseudo-counts |
| 21 Bayesian learning | any of the above + prior | — | the whole posterior, by MCMC |

The non-probabilistic exceptions — k-NN, decision trees, k-means, SVMs — are flagged as such in their modules, each with a
probabilistic cousin.

### 📘 Lecture notes — the two-step recipe, generalisation and what you should know

**The recipe (regression lecture).** Wish to learn $f: X \to Y$ from training data $\{\langle\mathbf x^1, y^1\rangle,
\dots, \langle\mathbf x^N, y^N\rangle\}$. **Step 1:** choose a parameterised form for $P(Y\mid X;\theta)$ (or $P(X;\theta)$
for unsupervised learning). **Step 2:** derive the learning algorithm as the MLE or the MAP estimate of $\theta$.

*Examples:* linear regression — $P(y\mid\mathbf x) = \mathcal N(\mathbf x^\top\mathbf w, \sigma^2)$, MLE = least squares,
MAP with a Gaussian prior = ridge; naïve Bayes — $P(Y)\prod_i P(X_i\mid Y)$, MLE = count ratios, MAP = pseudo-counts;
logistic regression — MLE by gradient ascent.

**Generalisation.** Learn on training data, then evaluate on previously unseen data: scores computed on the training
data are optimistic, because the parameters (and any choices such as which gene to use) were tuned to them.

**Worked example (Section 9).** One gene, a Gaussian per class and $\pi = P(\text{AML})$: five MLE parameters
estimated on 38 training patients; prediction by Bayes' rule
$P(y = 1\mid x) = \pi\mathcal N(x;\mu_1,\sigma_1^2)/[\pi\mathcal N(x;\mu_1,\sigma_1^2) + (1-\pi)\mathcal N(x;\mu_0,\sigma_0^2)]$;
accuracy measured on the 34 test patients.

**What you should know (the introduction lecture's summary).** Supervised vs unsupervised learning; supervised
learning learns the functional mapping $f: X \to Y$, often modelling it with $P(Y\mid X)$ to represent uncertainty;
unsupervised learning learns the density $P(X)$ to extract interesting patterns from the data; MLE vs MAP.

> 🎓 **Exam tip.** When a new model appears in a later lecture, immediately ask: what is Step 1 (which distribution, which parameters)
    and what is Step 2 (closed-form MLE? iterative? which prior gives the MAP version?). Most derivation questions are this
    recipe plus the calculus of Module 04.

## 📌 Key takeaways

- **Supervised learning** learns $P(Y\mid X)$ from labelled pairs (regression if $Y$ is continuous, classification if
  discrete); **unsupervised learning** learns $P(X)$ from inputs alone.
- The **likelihood** $P(D\mid\theta)$ is the probability of the observed data as a function of the parameters; for
  i.i.d. data it is a product, and we maximise its **log**. It is not a probability distribution over $\theta$.
- **MLE** = $\arg\max_\theta P(D\mid\theta)$: count ratios for Bernoulli/categorical models ($\hat q = 4/7$ for
  H,H,T,T,H,H,T; allele counting $\sum g/2N$), the sample mean and the $1/N$ variance for a Gaussian (biased by $(N-1)/N$).
  Maximising a Gaussian likelihood = minimising squared error.
- **Bayes' rule for parameters**: posterior ∝ likelihood × prior; the evidence $P(D)$ does not move the maximum.
  **MAP** = $\arg\max_\theta P(D\mid\theta)P(\theta)$.
- A **Beta($a, b$)** prior is conjugate to Bernoulli data: posterior Beta($a + n_1$, $b + n_0$),
  $\hat q_{MAP} = (n_1 + a - 1)/(N + a + b - 2)$ — **imaginary examples** that cure zero counts (naïve Bayes).
- MAP trades a little **bias** for less **variance**: it wins with few data, its prior's influence fades as $N$ grows,
  and a strong wrong prior hurts. A Gaussian prior on a mean **shrinks** toward the prior mean — the ridge formula.
- Learning the **joint** and conditioning works for a few variables but explodes exponentially ($3^{30} \approx 2\times
  10^{14}$); **conditional independence** (naïve Bayes, Bayesian networks, HMMs) makes learning possible.
- **The two-step recipe**: (1) choose $P(Y\mid X;\theta)$ or $P(X;\theta)$, (2) estimate $\theta$ by MLE or MAP — and
  judge the result on data the model has not seen.

---
## ✅ Checkpoint

1. Five isolates are R, S, S, S, R. Write $L(q)$ and $\ell(q)$ and find $\hat q_{MLE}$.
   <details><summary>Answer</summary>$L(q) = q^2(1-q)^3$, $\ell(q) = 2\log q + 3\log(1-q)$, $\ell'(q) = 2/q - 3/(1-q) = 0 \Rightarrow \hat q = 2/5 = 0.4$.</details>
2. Give two reasons for maximising the log-likelihood instead of the likelihood.
   <details><summary>Answer</summary>(1) $\log$ is increasing, so the maximiser is the same, and the log turns a product of $N$ terms into a sum whose derivative is easy (Module 04). (2) Products of many probabilities underflow to 0 in floating point; sums of logs do not.</details>
3. Five people have genotypes 0, 1, 2, 1, 0 (copies of T). What is the MLE of the T-allele frequency?
   <details><summary>Answer</summary>$\hat p = \sum g/(2N) = 4/10 = 0.4$ (allele counting).</details>
4. For the expression values 2, 4, 6 compute the Gaussian MLEs $\hat\mu$ and $\hat\sigma^2$, and the unbiased variance.
   <details><summary>Answer</summary>$\hat\mu = 4$; $\hat\sigma^2 = (4 + 0 + 4)/3 = 8/3 \approx 2.67$; unbiased $s^2 = 8/2 = 4$.</details>
5. With a Beta(3, 3) prior and data of 4 heads and 3 tails, give the posterior, the MAP estimate and the posterior mean.
   <details><summary>Answer</summary>Posterior Beta(3 + 4, 3 + 3) = Beta(7, 6); MAP $= (7-1)/(13-2) = 6/11 \approx 0.545$; mean $= 7/13 \approx 0.538$; both are pulled from the MLE 4/7 ≈ 0.571 toward 0.5.</details>
6. The naïve-Bayes lecture writes the MAP estimate as $\pi = (\#(Y=1) + \alpha_0)/(\#\text{samples} + \alpha_0 + \beta_0)$. Which Beta prior corresponds to $\alpha_0 = \beta_0 = 1$, and what problem does it solve?
   <details><summary>Answer</summary>Beta(2, 2) (the pseudo-counts $\alpha_0, \beta_0$ are $a-1, b-1$; the formula is the posterior mode). It adds one imaginary example of each outcome — add-one/Laplace smoothing — so no estimated probability can be exactly 0 (naïve-Bayes "subtlety #1").</details>
7. You estimate a resistance frequency from 3 isolates and from 3,000 isolates. In which case does the choice between MLE and MAP matter, and why?
   <details><summary>Answer</summary>With 3 isolates: the MLE has huge variance (and is 0 a third of the time when $q = 0.3$), and a reasonable prior reduces the MSE a lot. With 3,000 isolates the likelihood dominates any moderate prior (worth a few imaginary isolates), so MLE ≈ MAP; only a very strong prior would still matter — and then it adds bias.</details>
8. A joint probability table over 20 binary variables: how many free parameters? And for $P(Y\mid X_1,\dots,X_{20})$ with binary $Y$?
   <details><summary>Answer</summary>$2^{20} - 1 = 1{,}048{,}575$ for the joint; $2^{20} = 1{,}048{,}576$ (one probability per combination of the $X$'s) for the conditional. Naïve Bayes would need $2\cdot20\cdot1 + 1 = 41$ for binary features.</details>

## 🧠 Questions & discussion

1. The i.i.d. assumption underlies every likelihood in this notebook. Give two biological situations in which it fails (e.g. isolates from one outbreak, cells from one patient, related individuals in a GWAS). What happens to the likelihood's apparent certainty when observations are correlated but treated as independent?
2. A colleague says: "MAP is subjective, so I only report MLEs." Where in a standard analysis pipeline (choice of model, features, preprocessing, thresholds) are equally subjective choices made? When is an explicit prior *more* honest?
3. The Beta(30, 30) prior hurt at large $N$ in Section 5. How would you choose a prior's strength in practice? Discuss previous studies, held-out data (Module 23) and hierarchical/empirical-Bayes ideas (learning the prior from many genes, as in Section 6).
4. The one-gene classifier of Section 9 scored 100 % on training and 94 % on test patients. List every way information from the test patients could accidentally leak into training in a real study, and how each would inflate the test score.
5. Learning the joint distribution answers any question, but explodes in size. Learning only $P(Y\mid X)$ (logistic regression, Module 09) answers fewer questions with fewer parameters. When would a biologist want the full generative model of the data rather than just a classifier?

## 🚀 Bonus — going further

- **The Beta–binomial predictive.** Under a Beta($a, b$) posterior, the probability of seeing $k$ resistant isolates among the next $m$ is a Beta–binomial distribution (`scipy.stats.betabinom`). Compare it with the binomial($m$, $\hat q_{MLE}$) plug-in for $m = 10$: which one admits more uncertainty, and why?
- **MAP is not invariant to reparameterisation.** Compute the MAP of the log-odds $\theta = \log(q/(1-q))$ under the prior induced by Beta(3, 7), and transform back. Do you get 0.40? (The MLE, in contrast, is invariant.)
- **Laplace's rule of succession.** With a flat prior, the predictive probability that the next trial succeeds after $n_1$ successes in $N$ trials is $(n_1 + 1)/(N + 2)$. Derive it from the posterior mean and discuss the "sunrise problem".
- **Empirical Bayes for many genes.** In Section 6, estimate $\tau$ (the spread of true effects) from the 2,000 observed means themselves using $\operatorname{Var}(\bar x) = \tau^2 + \sigma^2/N$, plug it in as $\sigma_0$, and compare the MSE with the oracle choice.
- **More in this repository.** The [Fundamentals of learning and uncertainty](../../FundamentalsLearningUncertainty/) tutorial revisits supervised vs unsupervised learning, Gaussian MLE and MAP with posterior landscapes and asymptotic convergence; the [Generative vs discriminative classification](../../GenerativeDiscriminativeClassification/) tutorial continues Section 9 with Gaussian naïve Bayes vs logistic regression on breast-cancer data.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module:            05 · Likelihood, MLE and MAP
Input:             SIMULATED isolates (R,R,S,S,R,R,S), genotypes, replicate studies, fold changes, cohort;
                   SIMULATED leukaemia data (CD33, TCF3, 38/34 split); typed-in six-person genotype table
                   (naïve-Bayes lecture) and census joint table (introduction lecture)
Output:            likelihood curves, MLEs, Beta posteriors, MAP estimates, bias/variance/MSE tables, conditional tables,
                   a one-gene classifier's training and test accuracy
Method:            i.i.d. likelihood → log → derivative = 0 (or grid search); Bayes' rule with conjugate priors;
                   replicate simulation; counting + conditioning; generative classification by Bayes' rule
Key equation:      q̂_MLE = n1/N;   q̂_MAP = (n1 + a − 1)/(N + a + b − 2);   posterior ∝ likelihood × prior
Key parameters:    prior (a, b) or σ0; sample size N; number of variables in a joint table
What changed when I modified …  (e.g. Beta(3,7) → Beta(30,30); N = 3 → 300; 3 → 5 expression bins)
One failure mode:  MLE = 0 for an unseen category (zero counts) → a naïve-Bayes product collapses to 0
Connection to the 02-620 topics (which idea of the lecture this illustrates):
                   introduction (supervised P(Y|X) vs unsupervised P(X), joint then conditioning, MLE vs MAP, Bayes' rule);
                   naïve Bayes (MLE = counting, zero counts, imaginary examples); regression (two-step recipe, MAP = ridge)
```

## 📚 References

- Carnegie Mellon University 02-620 *Machine Learning for Scientists* (course design S. Kim; instructors M. Xu and M. Zhang) — this module teaches the material of its introductory lecture (supervised and unsupervised learning, MLE and MAP) and the estimation parts of its lectures on naïve Bayes and regression.
- Moore A. W. *Probabilistic and Bayesian Analytics* (tutorial, Carnegie Mellon University) — source of the gender/hours/wealth joint-distribution example (from US census records) and of the regressor/classifier/density-estimator overview.
- Murphy K. P. (2012) *Machine Learning: A Probabilistic Perspective*. MIT Press — §3.3 (the beta-binomial model: likelihood, prior, posterior, MAP, posterior mean, predictive), §3.4 (Dirichlet-multinomial), §4.1 and §4.6 (Gaussian MLE and inference with a Gaussian prior), §6.4 (bias, variance and MSE of estimators).
- Mitchell T. M. (1997) *Machine Learning*. McGraw-Hill — §6.2–6.4 (Bayes theorem, MAP and ML hypotheses), and the chapter "Generative and discriminative classifiers: naive Bayes and logistic regression" of the 2nd-edition drafts (MLE/MAP for naïve Bayes with imaginary examples).
- Hastie T., Tibshirani R., Friedman J. (2009) *The Elements of Statistical Learning*, 2nd ed. Springer — §2.6.3 and §8.2 (maximum likelihood), §8.3 (Bayesian methods), §7.2 (training vs test error).
- Golub T. R. *et al.* (1999) Molecular classification of cancer: class discovery and class prediction by gene expression monitoring. *Science* 286:531–537 (the study the simulated leukaemia data imitate).
- Dudoit S., Fridlyand J., Speed T. P. (2002) Comparison of discrimination methods for the classification of tumors using gene expression data. *Journal of the American Statistical Association* 97:77–87 (the preprocessing that the simulated leukaemia data imitate).
- Ambroise C., McLachlan G. J. (2002) Selection bias in gene extraction on the basis of microarray gene-expression data. *Proceedings of the National Academy of Sciences USA* 99:6562–6566.
- Domingos P., Pazzani M. (1997) On the optimality of the simple Bayesian classifier under zero-one loss. *Machine Learning* 29:103–130 (why naïve Bayes often classifies well despite its assumption).

## ➡️ Next

**[Module 06 · 📈 Linear regression and genome-wide association](06_linear_regression_gwas.ipynb)** — the first full
application of the two-step recipe: a Gaussian model for a quantitative trait given SNP genotypes, maximum conditional
likelihood = least squares, the normal equations of Modules 03–04 on the simulated prostate-cancer data, and the $N < J$ failure
that sends us to MAP estimation (ridge) in Module 07.